# Test U · Astensione e indeterminatezza logica

Notebook Colab per Qwen3-8B, Mistral-Nemo-Instruct-2407, OLMo-2-1124-7B-Instruct e Llama 3.1 8B Instruct. Disegno basato sulla bozza del Test U dell’8 novembre 2026. I risultati del Test 6 erano già noti. Le storie sono generate con il nuovo seme 20261120, mantenendo i formati ADD, V1, V2 e V3.

1. **Astensione.** HU1 confronta il punteggio di C con quello di B nella stessa storia. W conta una vittoria se C è più alto e mezza in caso di pareggio. HU2 calibra una soglia su ADD e la applica a V1, V2 e, come controllo descrittivo, V3. L’accuratezza bilanciata (BA) è la media di sensibilità sui C e specificità su A e B.
2. **Controllo su «or».** HO1 sostituisce *and* con *or* nella frase neutra in A. HO2 confronta *false, true* con *false or true* in B. Le due condizioni C permettono i controlli descrittivi. Il confronto B_set contro A_and può risentire della lunghezza. L’analisi separa anche la famiglia OR. La direzione SQuAD si ricostruisce con 600 prompt di addestramento e 400 di verifica. Sugli esempi logici si conservano solo i punteggi scalari.
3. **Polarità.** Screening A/B su 60 storie SCR, con tre candidati. Serve almeno 0,90 di accuratezza sia sui casi attesi YES sia su quelli attesi NO. In caso di parità si usa l’ordine N1, N2, N3. Solo il candidato selezionato e la formulazione granted sono valutati su storie POL distinte.

**Decisioni fissate.** W ≥ 0,60 per HU1, HO1, HO2 e HP1. ΔBA ≥ 0,05 per HU2. Per confermare, il limite inferiore dell’intervallo bootstrap deve superare il riferimento nullo. Si ricampionano le storie 2.000 volte. Per HU2 l’intervallo è congiunto: in ogni replica si ricampionano anche le storie di ADD e si ricalcola la soglia (l’intervallo condizionato alla soglia fissa si riporta come descrittivo). In caso di parità nella calibrazione si sceglie la soglia più alta. Le decisioni sono per modello e formato, senza correzione multipla. La sintesi per la libreria usa almeno 3 conferme sui 4 modelli previsti per lo stesso formato. Il numero di modelli completati è esplicito.

**Come partire.** In Colab seleziona una GPU A100 e lascia MODALITA = "prova" e MODELLI = ["qwen"]. La prova è ridotta e serve a controllare il funzionamento. Imposta AVVIA_GPU = True per lanciare la prova. Per la raccolta completa imposta MODALITA = "completo" e MODELLI = ["qwen", "mistral", "olmo", "llama"]. Drive conserva i blocchi completati e lo ZIP dopo ogni modello. Per Llama occorrono l’accesso al checkpoint e il Secret HF_TOKEN autorizzato per il notebook. Se il token o l’accesso mancano, Llama viene indicato come saltato. La prova riduce anche la selezione per la sonda SQuAD. La raccolta completa richiede circa 1 ora per modello (prompt uno alla volta, come nei test 4–6). La cella 11, facoltativa e solo CPU (parte solo con ESEGUI_COSENI = True), controlla i coseni allo strato 36 di Qwen del test 7. Il controllo numerico completo della sonda si esegue in modalità completa.

**Riferimenti di esecuzione** · [Template di chat Transformers](https://huggingface.co/docs/transformers/chat_templating) · [Qwen3-8B](https://huggingface.co/Qwen/Qwen3-8B) · [Accesso a Llama 3.1](https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct) · [SQuAD 2.0](https://huggingface.co/datasets/rajpurkar/squad_v2)

In [ ]:
# 1. Configurazione. Il preflight dati si può eseguire anche su CPU.
MODALITA = "prova"  # "prova" oppure "completo"
MODELLI = ["qwen"]  # raccolta completa: ["qwen", "mistral", "olmo", "llama"]
AVVIA_GPU = False
USA_DRIVE = True
BATCH_SIZE = 1        # 1 come nei test 4–6: nessun padding, stesse condizioni numeriche del controllo con il test 6
SQUAD_BATCH_SIZE = 1
# transformers: si usa la versione già installata in Colab, come nei test 4–7 (finisce nel manifest di ogni modello).


In [ ]:
import base64
import hashlib
import json
import os
import re
import zlib
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

LABELS = ("YES", "NO", "UNKNOWN")
FLIP = {"YES": "NO", "NO": "YES", "UNKNOWN": "UNKNOWN"}
ORIGINAL_SETS = ("ADD", "V1", "V2", "V3")
N_BOOT = 2000
SEED = 20261108

def encode(obj):
    return json.dumps(obj, sort_keys=True, ensure_ascii=False, allow_nan=False).encode("utf-8")

def fingerprint(obj):
    return hashlib.sha256(encode(obj)).hexdigest()

def write_json(path, obj):
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_bytes(encode(obj))
    os.replace(tmp, path)

def unpack(blob):
    return json.loads(zlib.decompress(base64.b64decode("".join(blob.split()))))

def subset(rows, limit):
    if limit is None:
        return rows
    kept = {}
    output = []
    for r in rows:
        ids = kept.setdefault((r["set"], r["family"]), set())
        if r["story_id"] in ids or len(ids) < limit:
            ids.add(r["story_id"])
            output.append(r)
    return output

def identify(rows):
    return [dict(row, row_id=f'{row["set"]}|{row["pair_key"]}|{row["kind"]}') for row in rows]

def parse_label(text):
    match = re.match(r"""^\s*[*#`"']*\s*(YES|NO|UNKNOWN)\b""", text, re.IGNORECASE)
    return match.group(1).upper() if match else "ALTRO"

def accuracy_by_label(rows):
    t = pd.DataFrame(rows)
    t["correct"] = t.parsed == t.condition
    group = t.groupby(["polarity", "condition"]).correct.agg(["mean", "size"])
    output = []
    for p in t.polarity.unique():
        yes, no = (group.loc[(p, label)] for label in ("YES", "NO"))
        output.append(dict(polarity=p, acc_yes=float(yes["mean"]), n_yes=int(yes["size"]),
                           acc_no=float(no["mean"]), n_no=int(no["size"]),
                           minimo=float(min(yes["mean"], no["mean"]))))
    return output

def pick_candidate(screen):
    by_candidate = {r["polarity"]: r["minimo"] for r in screen}
    best = max(CANDIDATES, key=by_candidate.__getitem__)
    return best if by_candidate[best] >= .90 else None

def bootstrap_story(values, smoke):
    per_story = values.groupby(level="story_id").mean().to_numpy()
    n = len(per_story)
    if not n:
        return dict(storie=0, stima=None, ci95=None)
    ci = None
    if not smoke and n > 1:
        rng = np.random.default_rng(SEED)
        b = per_story[rng.integers(0, n, (N_BOOT, n))].mean(axis=1)
        ci = np.quantile(b, [.025, .975]).tolist()
    return dict(storie=n, stima=float(per_story.mean()), ci95=ci)

In [ ]:
import itertools, json, hashlib, collections
import numpy as np
from pathlib import Path
DATA_SEED = 20261120
NOUNS = ['kiln', 'lantern', 'canoe', 'ledger', 'trolley', 'beacon', 'quiver', 'easel', 'turbine', 'hammock', 'compass', 'tripod',
         'satchel', 'kettle', 'anvil', 'barrel', 'sundial', 'telescope', 'harp', 'wagon', 'loom', 'drum', 'saddle', 'chimney',
         'furnace', 'orchard', 'pier', 'vault', 'archive', 'workshop', 'garage', 'stable', 'greenhouse', 'bakery', 'quarry',
         'harbor', 'mill', 'depot', 'pantry', 'cellar', 'attic', 'balcony', 'corridor', 'lobby', 'terrace', 'courtyard', 'gazebo',
         'pavilion', 'scanner', 'printer', 'router', 'battery', 'antenna', 'monitor', 'generator', 'conveyor', 'crane', 'pump',
         'ferry', 'tractor', 'bicycle', 'scooter', 'trailer', 'caravan', 'glider', 'kayak', 'raft', 'sled', 'carousel', 'fountain',
         'statue', 'mural', 'banner', 'flag', 'tent', 'cabin', 'shed', 'barn', 'silo', 'windmill', 'lighthouse', 'bridge', 'tunnel',
         'dam', 'well', 'cistern', 'boiler', 'radiator', 'heater', 'freezer', 'oven', 'stove', 'grill', 'blender', 'mixer',
         'toaster', 'kettledrum', 'piano', 'violin', 'cello', 'organ', 'trumpet', 'flute', 'clock', 'mirror', 'cabinet', 'dresser',
         'wardrobe', 'bench', 'stool', 'ladder', 'scaffold', 'elevator', 'escalator', 'turnstile', 'gate', 'fence', 'hedge']
PARTS = ['polished', 'inspected', 'unlocked', 'labeled', 'refueled', 'calibrated', 'swept', 'repainted', 'insured', 'registered',
         'sanitized', 'upgraded', 'serviced', 'cleared', 'certified', 'measured', 'tagged', 'photographed', 'reserved', 'approved']
INSTR = ('Use only the supplied fictional policy and facts. Each listed value is allowed; one actual value per field is chosen '
         'independently from its allowed values. Return exactly one of YES, NO, UNKNOWN on the first line, with no explanation.')
INSTR_PROSA = ('Read the short policy and the facts below. When a fact says that something may be true or false, either case is '
               'possible and you are not told which. Answer with a single word on the first line: YES, NO or UNKNOWN.')
POLS = {'granted': ('Is approval granted?', False), 'P1': ('Is approval refused?', True), 'P2': ('Is approval not granted?', True),
        'P3': ('Is the request rejected?', True)}           # (domanda, la risposta corretta si inverte?)
FAMS = {'ADD': ['AND', 'OR', 'IMPLIES'], 'V1': ['NAND', 'NOR', 'CONV'], 'V2': ['AND', 'OR', 'IMPLIES'], 'POL': ['AND', 'OR', 'IMPLIES'],
        'SCR': ['AND', 'OR', 'IMPLIES']}
N_ST = {'ADD': 120, 'V1': 200, 'V2': 200, 'V3': 60, 'POL': 100, 'SCR': 60}

def rule_fn(fam, neg):
    nb = (lambda b: not b) if neg else (lambda b: b)
    return {'AND': lambda a, b: a and nb(b), 'OR': lambda a, b: a or nb(b), 'IMPLIES': lambda a, b: (not a) or nb(b),
            'NAND': lambda a, b: not (a and nb(b)), 'NOR': lambda a, b: not (a or nb(b)), 'CONV': lambda a, b: (not nb(b)) or a}[fam]
def rule_sym(fam, neg, a, b):
    bb = f'NOT ({b})' if neg else b
    return {'AND': f'({a} AND {bb})', 'OR': f'({a} OR {bb})', 'IMPLIES': f'({a} IMPLIES {bb})',
            'NAND': f'NOT ({a} AND {bb})', 'NOR': f'NOT ({a} OR {bb})', 'CONV': f'({bb} IMPLIES {a})'}[fam]
def classify(f, a_vals, b_vals):
    outs = {bool(f(a, b)) for a in a_vals for b in b_vals}
    return 'YES' if outs == {True} else 'NO' if outs == {False} else 'UNKNOWN'
def flip(ans, do): return {'YES': 'NO', 'NO': 'YES', 'UNKNOWN': 'UNKNOWN'}[ans] if do else ans
fmt = lambda vals: ' or '.join(str(v).lower() for v in vals)
def phrase(name, val):                  # prosa: name = (sostantivo, participio). B e C differiscono solo per "true" / "false":
    n, p = name[0].replace('_', ' '), name[1]   # nessun "not" in più nei fatti (stesso numero di token, controllato in cella 4)
    if len(val) == 2: return f'It may be true or false that the {n} is {p}.'
    return f'It is {"true" if val[0] else "false"} that the {n} is {p}.'
def rule_prose(fam, neg, A, B):
    na, pa, nb_, pb = A[0], A[1], B[0], B[1]
    lit = lambda pos: f'the {nb_} is {pb}' if pos else f'the {nb_} is not {pb}'
    if fam == 'AND': return f'Approval is given if and only if both the {na} is {pa} and {lit(not neg)}.'
    if fam == 'OR': return f'Approval is given if and only if at least one of these holds: the {na} is {pa}; {lit(not neg)}.'
    if fam == 'IMPLIES': return f'Approval is given in every case except one: when the {na} is {pa} and {lit(neg)}.'
    raise ValueError(fam)

def build_all():
    rng = np.random.default_rng(DATA_SEED)
    names = [(n, p) for n in NOUNS for p in PARTS]; rng.shuffle(names); taken = set()
    def three():
        out = []
        for nm in names:
            if nm in taken or any(nm[0] == o[0] for o in out): continue
            out.append(nm)
            if len(out) == 3: break
        assert len(out) == 3, 'nomi finiti'
        taken.update(out); return out
    rows = []
    def add(**r): rows.append(r)
    for S in ('ADD', 'V1', 'V2', 'POL', 'SCR'):
        fams = FAMS[S]; n = N_ST[S]
        for i in range(n):
            fam = fams[i % len(fams)]; sid = f'{S}{i+1:03d}'
            A, B, Dn = three(); a, b, d = ('_'.join(A), '_'.join(B), '_'.join(Dn))
            d_val, a_single = bool(rng.integers(2)), bool(rng.integers(2))
            for neg in (False, True):
                f = rule_fn(fam, neg); variant = 'NOT' if neg else 'POS'
                bC = [bv for bv in (False, True) if classify(f, [False, True], [bv]) == 'UNKNOWN']
                bB = [bv for bv in (False, True) if classify(f, [False, True], [bv]) != 'UNKNOWN']
                assert len(bC) == len(bB) == 1; bC, bB = bC[0], bB[0]
                kinds = {'A': {a: [a_single], b: [bB], d: [d_val]}, 'B': {a: [False, True], b: [bB], d: [d_val]},
                         'C': {a: [False, True], b: [bC], d: [d_val]}}
                if S == 'SCR': kinds.pop('C')
                positions = [('ultima', [b, d, a])] if S in ('POL', 'SCR') else [('prima', [a, b, d]), ('ultima', [b, d, a])]
                pols = {'ADD': ['granted'], 'V1': ['granted'], 'V2': ['granted'], 'POL': ['granted', 'P1', 'P2', 'P3'],
                        'SCR': ['P1', 'P2', 'P3']}[S]
                for kind, al in kinds.items():
                    gold = classify(f, al[a], al[b]); assert (gold == 'UNKNOWN') == (kind == 'C')
                    for position, order in positions:
                        for pol in pols:
                            q, inv = POLS[pol]
                            if S == 'V2':
                                nm = {a: A, b: B, d: Dn}
                                lines = [phrase(nm[k], al[k]) for k in order]
                                sp = lambda T: (T[0].replace('_', ' '), T[1])
                                prompt = INSTR_PROSA + '\n\nPolicy: ' + rule_prose(fam, neg, sp(A), sp(B)) + ' No other rule applies.' \
                                         + '\nFacts:\n' + '\n'.join(lines) + '\n\nQuestion: Is approval given?'
                            else:
                                lines = [f'{k} = {fmt(al[k])}.' for k in order]
                                expr = rule_sym(fam, neg, a, b)
                                if pol == 'P3':
                                    rule = f'In this fictional policy, the request is rejected if and only if NOT ({expr}). No other rule applies.'
                                else:
                                    rule = f'In this fictional policy, approval is granted if and only if {expr}. No other approval rule applies.'
                                prompt = INSTR + '\n\n[R1] ' + rule + '\n[R2] ' + '\n'.join(lines) + '\n\nQuestion: ' + q
                            add(set=S, story_id=sid, family=fam, variant=variant, kind=kind, position=position, polarity=pol,
                                condition=flip(gold, inv), b_value=str(al[b][0]).lower(), fact_lines=lines, prompt=prompt,
                                pair_key=f'{sid}|{variant}|{position}|{pol}')
    for i in range(N_ST['V3']):                      # numerici: una sola variabile x (riga incerta) + distrattore
        sid = f'V3{i+1:03d}'; X, Dn = three()[:2]; x, d = '_'.join(X), '_'.join(Dn); t = 300 + 7 * i; d_val = bool(rng.integers(2))
        for variant, op, f in (('GE', 'at least', lambda v: v >= t), ('LE', 'at most', lambda v: v <= t)):
            vals = {'A': [t + 2] if variant == 'GE' else [t - 2], 'B': [t + 1, t + 2] if variant == 'GE' else [t - 2, t - 1],
                    'C': [t - 1, t] if variant == 'GE' else [t, t + 1]}
            for kind, xv in vals.items():
                outs = {f(v) for v in xv}; gold = 'YES' if outs == {True} else 'NO' if outs == {False} else 'UNKNOWN'
                assert (gold == 'UNKNOWN') == (kind == 'C')
                for position, order in (('prima', [x, d]), ('ultima', [d, x])):
                    al = {x: xv, d: [d_val]}; lines = [f'{k} = {fmt(al[k])}.' for k in order]
                    rule = f'In this fictional policy, approval is granted if and only if {x} is {op} {t}. No other approval rule applies.'
                    add(set='V3', story_id=sid, family='NUM', variant=variant, kind=kind, position=position, polarity='granted',
                        condition=gold, b_value='', fact_lines=lines,
                        prompt=INSTR + '\n\n[R1] ' + rule + '\n[R2] ' + '\n'.join(lines) + '\n\nQuestion: Is approval granted?',
                        pair_key=f'{sid}|{variant}|{position}|granted')
    return rows

In [ ]:
# Funzioni per il protocollo Test U.
QUESTIONS = {
    "granted": "Is approval granted?",
    "N1": "Is approval refused?",
    "N2": "Is approval refused?",
    "N3": "Is approval denied?",
}
CANDIDATES = ("N1", "N2", "N3")
EXAMPLE = (
    "Worked example only. In a separate policy, approval is granted if and only if badge_valid is true. "
    "If badge_valid = true, approval is granted, so the answer to 'Is approval refused?' is NO. "
    "Now apply only the policy and facts below."
)
NEUTRAL = "No other approval rule, written and unwritten, applies."

def question_variant(row, candidate):
    prefix, sep, old = row["prompt"].rpartition("\n\nQuestion: ")
    assert sep and old == QUESTIONS["granted"]
    if candidate != "granted":
        term = "denied" if candidate == "N3" else "refused"
        boundary = "\n[R2] "
        assert prefix.count(boundary) == 1
        prefix = prefix.replace(boundary, f" Approval is {term} exactly when it is not granted." + boundary)
        if candidate == "N2":
            prefix = prefix.replace("\n\n[R1] ", "\n\n" + EXAMPLE + "\n\n[R1] ", 1)
    return dict(row, prompt=prefix + sep + QUESTIONS[candidate], polarity=candidate,
                condition=FLIP[row["condition"]] if candidate != "granted" else row["condition"],
                pair_key=row["pair_key"].rsplit("|", 1)[0] + "|" + candidate)

def restore_granted(row):
    old = "Is approval refused?"
    assert row["prompt"].endswith("Question: " + old) and row["polarity"] == "P1"
    return dict(row, polarity="granted", condition=FLIP[row["condition"]],
                prompt=row["prompt"][:-len(old)] + QUESTIONS["granted"],
                pair_key=row["pair_key"].rsplit("|", 1)[0] + "|granted")

def lexical_rows(benchmark):
    rows = []
    for row in benchmark:
        if row["set"] != "ADD" or row["position"] != "ultima":
            continue
        stem = row["prompt"].replace("No other approval rule applies.", NEUTRAL, 1)
        assert NEUTRAL in stem
        if row["kind"] == "A":
            variants = {"A_and": stem, "A_or": stem.replace(NEUTRAL, NEUTRAL.replace("and", "or"), 1)}
        else:
            label = row["kind"]
            assert stem.count("= false or true.") == 1
            variants = {f"{label}_set": stem.replace("= false or true.", "= false, true.", 1), f"{label}_or": stem}
        for case, prompt in variants.items():
            rows.append(dict(row, set="O", lex_case=case, prompt=prompt,
                             row_id=f"O|{row['pair_key']}|{case}"))
    return rows

def make_inputs(dataset, smoke):
    rows = subset(dataset, 3 if smoke else None)
    benchmark = identify([r for r in rows if r["set"] in ORIGINAL_SETS])
    scr = [restore_granted(r) for r in rows if r["set"] == "SCR" and r["polarity"] == "P1"]
    screen = identify([question_variant(r, p) for r in scr for p in CANDIDATES])
    holdout = [r for r in rows if r["set"] == "POL" and r["polarity"] == "granted"]
    lexical = lexical_rows(benchmark)
    assert not ({r["story_id"] for r in screen} & {r["story_id"] for r in holdout})
    for data in (benchmark, screen, lexical):
        assert data and len({r["row_id"] for r in data}) == len(data)
    groups = pd.DataFrame(benchmark).groupby("pair_key")
    assert (groups.kind.nunique() == 3).all()
    assert all((r["kind"] == "C") == (r["condition"] == "UNKNOWN") for r in benchmark)
    return benchmark, screen, holdout, lexical

def filter_length(rows, lengths, lexical=False):
    t = pd.DataFrame(rows)
    t["prompt_tokens"] = lengths
    dropped = []
    if lexical:
        for (_, block) in t.groupby("pair_key"):
            for a, b in (("A_and", "A_or"), ("B_set", "B_or"), ("C_set", "C_or")):
                pair = block[block.lex_case.isin([a, b])]
                assert len(pair) == 2
                if pair.prompt_tokens.nunique() != 1:
                    dropped.extend(pair.row_id.tolist())
    else:
        relevant = t[t.kind.isin(["B", "C"]) & t["set"].ne("V3")]
        for _, block in relevant.groupby("pair_key"):
            assert len(block) == 2
            if block.prompt_tokens.nunique() != 1:
                dropped.extend(block.row_id.tolist())
    removed = set(dropped)
    return [r for r in rows if r["row_id"] not in removed], t[t.row_id.isin(removed)].to_dict("records")

def verdict(stat, null, minimum, smoke=False):
    if smoke:
        return "PROVA TECNICA"
    if stat["ci95"] is None:
        return "NON VALUTABILE"
    lo, hi = stat["ci95"]
    if lo > null and stat["stima"] >= minimum:
        return "CONFERMATA"
    return "NON CONFERMATA" if hi <= null else "NON CONCLUSIVA"

def compare_pairs(table, score, left, right, key, smoke):
    pairs = table.pivot(index=["story_id", "pair_key"], columns=key, values=score)
    clean = pairs.reindex(columns=[left, right]).dropna()
    wins = (clean[left] > clean[right]).astype(float) + .5 * (clean[left] == clean[right])
    result = bootstrap_story(wins, smoke)
    result["coppie"] = len(clean)
    return result

def permutation_w(paired, smoke):
    if paired.empty or smoke:
        return None
    x = paired.groupby(level="story_id").mean().to_numpy() - .5
    rng = np.random.default_rng(SEED)
    shuffled = (rng.choice([-1, 1], size=(2000, len(x))) * x).mean(axis=1)
    return float((1 + np.count_nonzero(shuffled >= x.mean() - 1e-12)) / (1 + len(shuffled)))

def with_scores(rows):
    t = pd.DataFrame(rows)
    probs = t[[f"p_{k}" for k in LABELS]].to_numpy(float)
    normalized = probs / np.maximum(probs.sum(axis=1, keepdims=True), 1e-300)
    t["meno_p_massima"] = -normalized.max(axis=1)
    t["entropia"] = -(normalized * np.log(np.maximum(normalized, 1e-300))).sum(axis=1)
    t["is_c"] = t.kind.eq("C")
    t["is_unknown"] = t.parsed.eq("UNKNOWN")
    return t

def benchmark_metrics(rows, smoke):
    t = with_scores(rows)
    t["ok"] = t.parsed.eq(t.condition)
    t["altro"] = t.parsed.eq("ALTRO")
    stats, detail = [], []
    for (scope, polarity), group in t.groupby(["set", "polarity"]):
        for keys, cell in group.groupby(["family", "variant", "position"]):
            for measure in ("p_UNKNOWN", "meno_p_massima", "entropia"):
                detail.append(dict(set=scope, polarity=polarity, family=keys[0], variant=keys[1], position=keys[2],
                                   misura=measure, **compare_pairs(cell, measure, "C", "B", "kind", smoke)))
        b_c = group[group.kind.isin(["B", "C"])]
        wide = b_c.pivot(index=["story_id", "pair_key"], columns="kind", values="p_UNKNOWN").reindex(columns=["B", "C"]).dropna()
        w = bootstrap_story((wide.C > wide.B).astype(float) + .5 * (wide.C == wide.B), smoke)
        w["coppie"] = len(wide)
        unanswered = group[(group.kind == "C") & ~group.is_unknown].pair_key
        without_abst = group[group.pair_key.isin(unanswered)]
        strict_w = compare_pairs(without_abst, "p_UNKNOWN", "C", "B", "kind", smoke)
        bc_full = b_c[b_c.pair_key.isin(wide.index.get_level_values("pair_key"))]
        auroc = float(roc_auc_score(bc_full.is_c, bc_full.p_UNKNOWN)) if len(bc_full) else None
        behavior = []
        for kind, k in group.groupby("kind"):
            behavior.append(dict(kind=kind, n=len(k), acc=float(k.ok.mean()), quota_unknown=float(k.is_unknown.mean()),
                                 n_unknown=int(k.is_unknown.sum()), n_altro=int(k.altro.sum()),
                                 n_yes=int(k.parsed.eq("YES").sum()), n_no=int(k.parsed.eq("NO").sum())))
        by_position = []                      # descrittivo: astensione generata per tipo e posizione della riga incerta
        for (kind, position), k in group.groupby(["kind", "position"]):
            by_position.append(dict(kind=kind, position=position, n=len(k), quota_unknown=float(k.is_unknown.mean()),
                                    p_UNKNOWN_mediana=float(k.p_UNKNOWN.median())))
        alternative = {measure: compare_pairs(group, measure, "C", "B", "kind", smoke)
                       for measure in ("meno_p_massima", "entropia")}
        stats.append(dict(set=scope, polarity=polarity, W=w, AUROC=auroc, comportamento=behavior, comportamento_per_posizione=by_position,
                          W_senza_UNKNOWN_generato=strict_w, alternative=alternative,
                          p_perm=permutation_w((wide.C > wide.B).astype(float) + .5 * (wide.C == wide.B), smoke),
                          esito="DESCRITTIVA" if scope == "V3" or (scope == "POL" and polarity == "granted")
                          else verdict(w, .5, .60, smoke)))
    return stats, detail

def fit_threshold(add, score="p_UNKNOWN"):
    grid = add.groupby(score).is_c.agg(["sum", "size"]).sort_index(ascending=False)
    positive = grid["sum"].to_numpy()
    negative = grid["size"].to_numpy() - positive
    assert positive.sum() > 0 and negative.sum() > 0
    tpr = np.r_[0, positive.cumsum()] / positive.sum()
    fpr = np.r_[0, negative.cumsum()] / negative.sum()
    tau = np.r_[grid.index.to_numpy(dtype=float), np.nextafter(float(grid.index[-1]), -np.inf)]
    ba = .5 * (tpr + 1 - fpr)
    best = int(np.argmax(np.round(ba, 12)))
    return float(tau[best]), float(ba[best])

def transferred_threshold(rows, smoke, score="p_UNKNOWN"):
    # HU2. Intervallo (decide): bootstrap congiunto. In ogni replica si ricampionano le storie di ADD e si ricalcola la soglia,
    # e si ricampionano in modo indipendente le storie dell'insieme di verifica. Così l'intervallo include anche l'incertezza
    # della soglia. L'intervallo condizionato alla soglia fissata su tutto ADD si riporta come descrittivo (ci95_condizionato).
    data = with_scores(rows)
    add = data[data["set"] == "ADD"].reset_index(drop=True)
    tau, ba_add = fit_threshold(add, score)
    taus = None
    if not smoke:
        rng_add = np.random.default_rng(SEED + 1)
        rows_by_story = [np.asarray(g) for g in add.groupby("story_id").indices.values()]
        taus = np.empty(N_BOOT)
        for b in range(N_BOOT):
            pick = rng_add.integers(0, len(rows_by_story), len(rows_by_story))
            taus[b] = fit_threshold(add.iloc[np.concatenate([rows_by_story[i] for i in pick])], score)[0]
    out = []
    for name in ("V1", "V2", "V3"):
        t = data[data["set"] == name].copy()
        t["rule"] = t[score].gt(tau)
        t["hit_rule"] = (t.rule == t.is_c).astype(int)
        t["hit_model"] = (t.is_unknown == t.is_c).astype(int)
        groups = t.groupby(["story_id", "is_c"]).agg(n=("is_c", "size"), rule=("hit_rule", "sum"), model=("hit_model", "sum"))
        matrix = groups.unstack(fill_value=0)
        ci_cond = None
        if matrix.empty or not {False, True}.issubset(matrix["n"].columns):
            result = dict(storie=len(matrix), stima=None, ci95=None)
            ba_rule = ba_model = None
        else:
            n, hit_rule, hit_model = (matrix[k].to_numpy() for k in ("n", "rule", "model"))
            ba_rule = float((hit_rule.sum(axis=0) / n.sum(axis=0)).mean())
            ba_model = float((hit_model.sum(axis=0) / n.sum(axis=0)).mean())
            ci = None
            if not smoke and len(n) > 1:
                rng = np.random.default_rng(SEED)
                indices = rng.integers(0, len(n), (N_BOOT, len(n)))
                denom = n[indices].sum(axis=1)
                good = (denom > 0).all(axis=1)
                delta = ((hit_rule - hit_model)[indices].sum(axis=1)[good] / denom[good]).mean(axis=1)
                if len(delta):
                    ci_cond = np.quantile(delta, [.025, .975]).tolist()
                # bootstrap congiunto: stessa estrazione delle storie di verifica, soglia della replica b
                stories = matrix.index.to_numpy()
                pos = {s: i for i, s in enumerate(stories)}
                srow = t.story_id.map(pos).to_numpy()
                cls = t.is_c.to_numpy().astype(int)
                sc_ = t[score].to_numpy(float)
                cell = srow * 2 + cls
                deltas = []
                for b in range(N_BOOT):
                    if not good[b]:
                        continue
                    hr = np.bincount(cell, weights=((sc_ > taus[b]) == cls.astype(bool)), minlength=2 * len(stories)).reshape(-1, 2)
                    num = (hr - hit_model)[indices[b]].sum(axis=0)
                    deltas.append((num / denom[b]).mean())
                if deltas:
                    ci = np.quantile(deltas, [.025, .975]).tolist()
            result = dict(storie=len(n), stima=ba_rule - ba_model, ci95=ci, ci95_condizionato=ci_cond)
        out.append(dict(set=name, tau=tau, BA_ADD_in_sample=ba_add, BA_soglia=ba_rule, BA_modello=ba_model,
                        tau_bootstrap_ci95=None if taus is None else np.quantile(taus, [.025, .975]).tolist(),
                        **result, esito="DESCRITTIVA" if name == "V3" or score != "p_UNKNOWN" else verdict(result, 0, .05, smoke)))
    return out

def lexical_metrics(rows, smoke):
    columns = ["row_id", "story_id", "pair_key", "family", "lex_case", "score_squad", "p_UNKNOWN", "prompt_tokens", "parsed"]
    t = pd.DataFrame(rows, columns=columns)
    out = []
    scopes = [("tutte", t), ("OR", t[t.family == "OR"]), ("senza_OR", t[t.family != "OR"])]
    for name, group in scopes:
        lengths = group.pivot(index=["story_id", "pair_key"], columns="lex_case", values="prompt_tokens")
        for hyp, a, b in (("HO1", "A_or", "A_and"), ("HO2", "B_or", "B_set"),
                           ("due_valori", "B_set", "A_and"), ("C_vs_B_set", "C_set", "B_set"), ("C_vs_B_or", "C_or", "B_or")):
            diff = (lengths.reindex(columns=[a, b])[a] - lengths.reindex(columns=[a, b])[b]).dropna()
            for score in ("score_squad", "p_UNKNOWN"):
                comparison = group
                if hyp.startswith("C_vs_B"):
                    same_length_keys = diff[diff == 0].index.get_level_values("pair_key")
                    comparison = group[group.pair_key.isin(same_length_keys)]
                stat = compare_pairs(comparison, score, a, b, "lex_case", smoke)
                out.append(dict(scope=name, ipotesi=hyp, misura=score, confronto=f"{a} > {b}",
                                delta_token_medio=float(diff.mean()) if len(diff) else None,
                                **stat, esito=verdict(stat, .5, .60, smoke)
                                if hyp in ("HO1", "HO2") and score == "score_squad" and name == "tutte" else "DESCRITTIVA"))
    gen = []
    for name, group in t[t.lex_case.isin(["C_or", "C_set"])].groupby("lex_case"):
        gen.append(dict(lex_case=name, n=len(group), quota_UNKNOWN=float(group.parsed.eq("UNKNOWN").mean())))
    return out, gen

In [ ]:
MODEL_CONFIG = {'qwen': {'id': 'Qwen/Qwen3-8B', 'sha': 'b968826d9c46dd6066d109eabc6255188de91218', 'layers': [23, 26, 30, 33], 'label_ids': {'YES': [7414, 9454, 9693, 9834, 14004, 14080], 'NO': [902, 2152, 2308, 2753, 5664, 8996], 'UNKNOWN': [9788, 13790, 16088, 21693, 51378, 61395]}}, 'mistral': {'id': 'mistralai/Mistral-Nemo-Instruct-2407', 'sha': '04d8a90549d23fc6bd7f642064003592df51e9b3', 'layers': [12, 14, 17, 20, 23, 26], 'label_ids': {'YES': [13059, 13830, 14842, 16860, 51935, 57445], 'NO': [1836, 2649, 3501, 4753, 14089, 16071], 'UNKNOWN': [8929, 14183, 35988, 43048, 51641, 117195]}}, 'olmo': {'id': 'allenai/OLMo-2-1124-7B-Instruct', 'sha': '470b1fba1ae01581f270116362ee4aa1b97f4c84', 'layers': [11, 14, 17, 20, 23, 26], 'label_ids': {'YES': [7566, 9642, 9891, 10035, 14331, 14410], 'NO': [912, 2201, 2360, 2822, 5782, 9173], 'UNKNOWN': [9987, 14109, 16476, 22435, 52478, 62495]}}, 'llama': {'id': 'meta-llama/Llama-3.1-8B-Instruct', 'sha': None}}
REFERENCE = {'squad': {'qwen': {'ids': ['56ddde6b9a695914005b962a', '57094b4f9928a814004714f8', '57113f83b654c5140001fc2d', '5725c28a271a42140099d14e', '57265aaf5951b619008f706e', '572683f95951b619008f7527', '5727515f708984140094dc15', '5728fd206aef05140015494d', '57299021af94a219006aa50b', '573368e54776f41900660a54', '5ad3de8b604f3c001a3ff469', '5ad26586d7d075001a4291b5', '5ad25b1cd7d075001a428e67', '5a83a59be60761001a2eb84c', '5a2ebb5fa83784001a7d244b', '5a6ceb7a4eec6b001a80a6d2', '5a1c6968b4fb5d001871461a', '5ad4acbd5b96ef001a109d78', '5ad2b882d7d075001a42a042', '5ad50f1c5b96ef001a10ab31'], 'projections': [[-5.470077541222387, -19.370739042511275, -50.20814929652002, -11.470840290592005], [-15.392857733647643, -33.08018170933117, -67.35186428350832, -57.535788896965386], [-9.22975666326676, -23.59060384278162, -56.483388034837176, -52.29017322558392], [-23.090113621878643, -39.015487263599866, -60.40009863532645, -46.569248870093226], [-21.067415998439195, -38.924934974820566, -84.06034239785292, -70.00686680693138], [-18.40825481833874, -31.804532842348884, -73.48842783091897, -70.43975194351637], [-14.827555748353417, -36.25445859870747, -66.43459540915495, -59.38688116975428], [-15.004780986925812, -27.997466022315965, -62.03209044172546, -47.74864976231041], [9.24186312503371, 40.96027524027146, 96.95636952263106, 226.90775377178068], [-23.365598421266363, -40.21166570147278, -85.646475408689, -54.93633541132213], [6.1996751100624845, 14.120687720173944, 12.241787144276902, 99.07086970949211], [30.85375389805669, 104.47258136466428, 172.70096097103055, 310.1030423821944], [0.0338980115474574, -5.666671478813038, -22.137553680239343, 30.100477226258143], [17.31980928484144, 58.46323320568483, 105.76149858690582, 221.02961893075684], [-3.367909245623624, -12.159675560521311, -40.90338055838379, -15.87103582691427], [38.25366339063992, 96.45658853768175, 162.14481641080755, 279.5551848789726], [24.429746133678808, 72.97110733253855, 113.93019282736614, 232.4570764321572], [14.870369123343764, 14.427940010727404, -1.697172473731353, 64.77936825611398], [11.301362837560236, 66.93544459483454, 122.99446225580476, 264.7696756947627], [17.161872451483358, 49.96907028388792, 117.05534933752614, 244.53431313218297]], 'auc': [0.9263, 0.9185749999999999, 0.9157249999999999, 0.90795]}, 'mistral': {'ids': ['56ddde6b9a695914005b962a', '57094b4f9928a814004714f8', '57113f83b654c5140001fc2d', '5725c28a271a42140099d14e', '57265aaf5951b619008f706e', '572683f95951b619008f7527', '5727515f708984140094dc15', '5728fd206aef05140015494d', '57299021af94a219006aa50b', '573368e54776f41900660a54', '5ad3de8b604f3c001a3ff469', '5ad26586d7d075001a4291b5', '5ad25b1cd7d075001a428e67', '5a83a59be60761001a2eb84c', '5a2ebb5fa83784001a7d244b', '5a6ceb7a4eec6b001a80a6d2', '5a1c6968b4fb5d001871461a', '5ad4acbd5b96ef001a109d78', '5ad2b882d7d075001a42a042', '5ad50f1c5b96ef001a10ab31'], 'projections': [[-2.2691444064986674, -1.0444595709071058, -2.8316129621304245, 0.739955234653018, -0.8091155702518228, 0.136305147110528], [-2.9965279473161157, -1.7371206099913383, -3.574219177145236, -0.0111432331863212, -0.2948280843057722, 2.3724054653519504], [-2.4816777246642165, -1.1972831653850535, -1.7278913438111705, 2.4180494044223133, 2.2901340824544807, 4.563405389487463], [-2.7398565633226597, -1.4671936230988052, -2.837786389175875, -0.6294770884447032, -2.234292197684086, -1.2655949989209123], [-3.074866129580184, -1.7572208800143163, -3.5085513204627787, 0.3456230243321787, -0.1818960195762479, 1.334455130518084], [-2.8761476095283363, -1.5303891378763017, -4.134968179006407, -0.2042504787348857, -0.6940607958168081, 1.255972707821626], [-2.710097697667276, -1.316870690743282, -3.162597704611935, -1.4647211635558528, -3.0044622957605727, -1.003505975625299], [-1.198031605181872, -0.1369184727214801, -2.801531238674794, -0.2943325180979568, -0.7511693842602043, 1.6999331582708908], [-1.399660698758403, 0.0221491025899227, -1.2589433181264629, 1.5840046032658557, -0.0507215182282912, 2.617578271043873], [-3.3799387722548566, -1.9547022432750416, -3.760700529969946, -0.3425976302872703, -1.572011892030312, -0.8976931335057541], [-1.8392317311050557, -0.4537939392255501, -1.5465481212145762, 4.2439039430537004, 6.560262831174853, 9.12977294189235], [-1.60062275516459, 0.1074537806354795, -0.4143349650760675, 7.899201931131377, 12.44519213142165, 17.685337614705443], [-2.432139948276594, -1.174860003140923, -2.2452797076098374, 6.9454676192105484, 9.816402665474186, 13.916619217163005], [-2.152878795371349, -0.5447390360168635, -1.7426701319192175, 3.708000018785872, 3.015036233590361, 5.052519158738139], [-1.1645686368991852, 0.1574814269097355, -1.8108393854979008, 2.7468997700179187, 1.9966995489419008, 3.0330309723982154], [0.6444646424910545, 2.3852093350307686, 2.182341774599656, 11.741834586760072, 16.653933670634693, 21.61181182455529], [0.1247141274374815, 1.7601093786931872, -0.1695641683904546, 6.089693820558225, 10.639222209520003, 14.684520378641851], [-1.4699939950785703, 0.2827780994901887, -0.5634896884083633, 4.161155769124568, 3.977323653878973, 5.564232412297814], [-2.752789718346593, -1.2787318150685982, -2.3284223647642386, 1.7451014835984413, 0.5753835930263878, 1.2367822502986652], [-2.623960692699546, -0.8899125945912877, -1.4654886357057286, 7.195840258779992, 10.994856513672428, 14.229024487921372]], 'auc': [0.6638999999999999, 0.7384, 0.8432000000000001, 0.9168000000000001, 0.90725, 0.902725]}, 'olmo': {'ids': ['56ddde6b9a695914005b962a', '57094b4f9928a814004714f8', '57113f83b654c5140001fc2d', '5725c28a271a42140099d14e', '57265aaf5951b619008f706e', '572683f95951b619008f7527', '5727515f708984140094dc15', '5728fd206aef05140015494d', '57299021af94a219006aa50b', '573368e54776f41900660a54', '5ad3de8b604f3c001a3ff469', '5ad26586d7d075001a4291b5', '5ad25b1cd7d075001a428e67', '5a83a59be60761001a2eb84c', '5a2ebb5fa83784001a7d244b', '5a6ceb7a4eec6b001a80a6d2', '5a1c6968b4fb5d001871461a', '5ad4acbd5b96ef001a109d78', '5ad2b882d7d075001a42a042', '5ad50f1c5b96ef001a10ab31'], 'projections': [[-0.2900606086694632, -1.1182152225340758, -2.747978953087177, -1.796589269527867, -0.7648445953097545, 0.8194210298270515], [-0.4947748691751165, -1.1170058744161553, -2.10582972701588, -1.6006777902925315, -1.0318251880720162, -0.1876309247064528], [-0.2728623418336143, -0.8136419643851798, -1.908239247601606, -1.1842806599867195, -0.4380942583959215, 0.5991363258486806], [-0.2572764168077114, -1.5442677032286507, -2.334109158871698, -1.702511586470638, -1.262543275361228, -0.1688887833352407], [-0.3494578735757964, -1.3182027695960872, -2.120529013310917, -1.2820365302603598, -0.8923696587083585, 0.2649197121380745], [-0.5950660706812253, -1.3437501694030325, -2.9417002855374434, -1.6741154556540478, -0.8808201266595441, 0.9077418991785996], [-0.4841715753979732, -1.3423319140122747, -2.657117210577644, -1.3616726476035383, -0.959163003434022, 0.0818121298732131], [-0.2509926348897598, -1.330850582972379, -2.193332449681752, -1.0617250296076717, -0.5096062321017203, 1.0450641190370618], [-0.1515950337178795, -1.0400360181743011, -2.102788695669826, -0.8265255944974554, 0.0097831505625142, 0.7259720605420681], [-0.4860095318390188, -1.4871171472647398, -3.721066592452121, -2.5382522853239786, -2.4825889514690056, -2.0805985626274306], [-0.1759667971878575, -0.2630559367062779, 1.61201753702556, 3.714408426448789, 3.742917290630104, 5.772816048894929], [-0.2682636049023715, -0.5515280922065899, -0.5214410625966892, 1.9521289722838, 3.502356154881556, 5.433244506039387], [-0.2967575466297217, -0.2664902553375943, 0.0401053441426944, 0.3978160372352914, 0.6618527025317622, 1.6921055530217062], [-0.2908867924807241, -0.5529511084515097, -0.337728539606498, 1.769450417215093, 3.449374741085515, 6.614078549349979], [-0.3773313719082795, -1.0654764322029924, -2.316179024904632, -0.9181518668612978, -0.1664316880212904, 1.228981834702048], [0.4212655727080097, 0.7593842035856901, 3.061236104924088, 6.6082948613993455, 9.133507779530651, 13.369150809203887], [-0.1153363483695305, -0.7049358971274948, 0.4945683904452678, 1.785002532976744, 3.2222923311853524, 5.809558839425405], [-0.2168604313823513, -0.4983332363297427, 0.5663430075794964, 1.942830961061346, 2.090006261692035, 2.2045005581292108], [-0.5717516168194022, -1.079177531975989, -1.9425763710255344, -1.0266627006758564, 0.0396633066866755, 2.86746655932797], [-0.4669293592736971, -1.2890715629333005, -0.9630525149918944, 0.4016172319271177, 1.2260677367574824, 2.477963197284926]], 'auc': [0.635025, 0.8207, 0.884275, 0.88885, 0.8907499999999999, 0.8723750000000001]}}, 'outputs': {'qwen': [{'prompt': 'Use only the supplied fictional policy and facts. Each listed value is allowed; one actual value per field is chosen independently from its allowed values. Return exactly one of YES, NO, UNKNOWN on the first line, with no explanation.\n\n[R1] In this fictional policy, approval is granted if and only if (canoe_photographed AND violin_registered). No other approval rule applies.\n[R2] canoe_photographed = true.\nviolin_registered = false.\nladder_insured = true.\n\nQuestion: Is approval granted?', 'kind': 'A', 'set': 'ADD', 'position': 'prima', 'expected_p_UNKNOWN': 1.2502275881232092e-09}, {'prompt': 'Use only the supplied fictional policy and facts. Each listed value is allowed; one actual value per field is chosen independently from its allowed values. Return exactly one of YES, NO, UNKNOWN on the first line, with no explanation.\n\n[R1] In this fictional policy, approval is granted if and only if (canoe_photographed AND violin_registered). No other approval rule applies.\n[R2] violin_registered = false.\nladder_insured = true.\ncanoe_photographed = true.\n\nQuestion: Is approval granted?', 'kind': 'A', 'set': 'ADD', 'position': 'ultima', 'expected_p_UNKNOWN': 6.102143530056782e-13}, {'prompt': 'Use only the supplied fictional policy and facts. Each listed value is allowed; one actual value per field is chosen independently from its allowed values. Return exactly one of YES, NO, UNKNOWN on the first line, with no explanation.\n\n[R1] In this fictional policy, approval is granted if and only if (canoe_photographed AND violin_registered). No other approval rule applies.\n[R2] canoe_photographed = false or true.\nviolin_registered = false.\nladder_insured = true.\n\nQuestion: Is approval granted?', 'kind': 'B', 'set': 'ADD', 'position': 'prima', 'expected_p_UNKNOWN': 1.316675379836596e-10}, {'prompt': 'Use only the supplied fictional policy and facts. Each listed value is allowed; one actual value per field is chosen independently from its allowed values. Return exactly one of YES, NO, UNKNOWN on the first line, with no explanation.\n\n[R1] In this fictional policy, approval is granted if and only if (canoe_photographed AND violin_registered). No other approval rule applies.\n[R2] violin_registered = false.\nladder_insured = true.\ncanoe_photographed = false or true.\n\nQuestion: Is approval granted?', 'kind': 'B', 'set': 'ADD', 'position': 'ultima', 'expected_p_UNKNOWN': 1.6702560969861224e-05}, {'prompt': 'Use only the supplied fictional policy and facts. Each listed value is allowed; one actual value per field is chosen independently from its allowed values. Return exactly one of YES, NO, UNKNOWN on the first line, with no explanation.\n\n[R1] In this fictional policy, approval is granted if and only if (canoe_photographed AND violin_registered). No other approval rule applies.\n[R2] canoe_photographed = false or true.\nviolin_registered = true.\nladder_insured = true.\n\nQuestion: Is approval granted?', 'kind': 'C', 'set': 'ADD', 'position': 'prima', 'expected_p_UNKNOWN': 4.5091660440255135e-12}, {'prompt': 'Use only the supplied fictional policy and facts. Each listed value is allowed; one actual value per field is chosen independently from its allowed values. Return exactly one of YES, NO, UNKNOWN on the first line, with no explanation.\n\n[R1] In this fictional policy, approval is granted if and only if (canoe_photographed AND violin_registered). No other approval rule applies.\n[R2] violin_registered = true.\nladder_insured = true.\ncanoe_photographed = false or true.\n\nQuestion: Is approval granted?', 'kind': 'C', 'set': 'ADD', 'position': 'ultima', 'expected_p_UNKNOWN': 3.721874236362055e-06}, {'prompt': 'Use only the supplied fictional policy and facts. Each listed value is allowed; one actual value per field is chosen independently from its allowed values. Return exactly one of YES, NO, UNKNOWN on the first line, with no explanation.\n\n[R1] In this fictional policy, approval is granted if and only if NOT (easel_insured AND greenhouse_cleared). No other approval rule applies.\n[R2] easel_insured = false.\ngreenhouse_cleared = false.\nescalator_repainted = false.\n\nQuestion: Is approval granted?', 'kind': 'A', 'set': 'V1', 'position': 'prima', 'expected_p_UNKNOWN': 1.2257417303374039e-11}, {'prompt': 'Use only the supplied fictional policy and facts. Each listed value is allowed; one actual value per field is chosen independently from its allowed values. Return exactly one of YES, NO, UNKNOWN on the first line, with no explanation.\n\n[R1] In this fictional policy, approval is granted if and only if NOT (easel_insured AND greenhouse_cleared). No other approval rule applies.\n[R2] greenhouse_cleared = false.\nescalator_repainted = false.\neasel_insured = false.\n\nQuestion: Is approval granted?', 'kind': 'A', 'set': 'V1', 'position': 'ultima', 'expected_p_UNKNOWN': 2.5949353776866246e-11}, {'prompt': 'Use only the supplied fictional policy and facts. Each listed value is allowed; one actual value per field is chosen independently from its allowed values. Return exactly one of YES, NO, UNKNOWN on the first line, with no explanation.\n\n[R1] In this fictional policy, approval is granted if and only if NOT (easel_insured AND greenhouse_cleared). No other approval rule applies.\n[R2] easel_insured = false or true.\ngreenhouse_cleared = false.\nescalator_repainted = false.\n\nQuestion: Is approval granted?', 'kind': 'B', 'set': 'V1', 'position': 'prima', 'expected_p_UNKNOWN': 5.211907572189034e-10}, {'prompt': 'Use only the supplied fictional policy and facts. Each listed value is allowed; one actual value per field is chosen independently from its allowed values. Return exactly one of YES, NO, UNKNOWN on the first line, with no explanation.\n\n[R1] In this fictional policy, approval is granted if and only if NOT (easel_insured AND greenhouse_cleared). No other approval rule applies.\n[R2] greenhouse_cleared = false.\nescalator_repainted = false.\neasel_insured = false or true.\n\nQuestion: Is approval granted?', 'kind': 'B', 'set': 'V1', 'position': 'ultima', 'expected_p_UNKNOWN': 4.364094863262835e-09}, {'prompt': 'Use only the supplied fictional policy and facts. Each listed value is allowed; one actual value per field is chosen independently from its allowed values. Return exactly one of YES, NO, UNKNOWN on the first line, with no explanation.\n\n[R1] In this fictional policy, approval is granted if and only if NOT (easel_insured AND greenhouse_cleared). No other approval rule applies.\n[R2] easel_insured = false or true.\ngreenhouse_cleared = true.\nescalator_repainted = false.\n\nQuestion: Is approval granted?', 'kind': 'C', 'set': 'V1', 'position': 'prima', 'expected_p_UNKNOWN': 5.4933439741500216e-11}, {'prompt': 'Use only the supplied fictional policy and facts. Each listed value is allowed; one actual value per field is chosen independently from its allowed values. Return exactly one of YES, NO, UNKNOWN on the first line, with no explanation.\n\n[R1] In this fictional policy, approval is granted if and only if NOT (easel_insured AND greenhouse_cleared). No other approval rule applies.\n[R2] greenhouse_cleared = true.\nescalator_repainted = false.\neasel_insured = false or true.\n\nQuestion: Is approval granted?', 'kind': 'C', 'set': 'V1', 'position': 'ultima', 'expected_p_UNKNOWN': 5.493274238266288e-11}, {'prompt': 'Read the short policy and the facts below. When a fact says that something may be true or false, either case is possible and you are not told which. Answer with a single word on the first line: YES, NO or UNKNOWN.\n\nPolicy: Approval is given if and only if both the dam is certified and the depot is photographed. No other rule applies.\nFacts:\nIt is true that the dam is certified.\nIt is false that the depot is photographed.\nIt is true that the gazebo is reserved.\n\nQuestion: Is approval given?', 'kind': 'A', 'set': 'V2', 'position': 'prima', 'expected_p_UNKNOWN': 4.785391411132878e-06}, {'prompt': 'Read the short policy and the facts below. When a fact says that something may be true or false, either case is possible and you are not told which. Answer with a single word on the first line: YES, NO or UNKNOWN.\n\nPolicy: Approval is given if and only if both the dam is certified and the depot is photographed. No other rule applies.\nFacts:\nIt is false that the depot is photographed.\nIt is true that the gazebo is reserved.\nIt is true that the dam is certified.\n\nQuestion: Is approval given?', 'kind': 'A', 'set': 'V2', 'position': 'ultima', 'expected_p_UNKNOWN': 0.0031671121250838}, {'prompt': 'Read the short policy and the facts below. When a fact says that something may be true or false, either case is possible and you are not told which. Answer with a single word on the first line: YES, NO or UNKNOWN.\n\nPolicy: Approval is given if and only if both the dam is certified and the depot is photographed. No other rule applies.\nFacts:\nIt may be true or false that the dam is certified.\nIt is false that the depot is photographed.\nIt is true that the gazebo is reserved.\n\nQuestion: Is approval given?', 'kind': 'B', 'set': 'V2', 'position': 'prima', 'expected_p_UNKNOWN': 0.0066933576017618}, {'prompt': 'Read the short policy and the facts below. When a fact says that something may be true or false, either case is possible and you are not told which. Answer with a single word on the first line: YES, NO or UNKNOWN.\n\nPolicy: Approval is given if and only if both the dam is certified and the depot is photographed. No other rule applies.\nFacts:\nIt is false that the depot is photographed.\nIt is true that the gazebo is reserved.\nIt may be true or false that the dam is certified.\n\nQuestion: Is approval given?', 'kind': 'B', 'set': 'V2', 'position': 'ultima', 'expected_p_UNKNOWN': 0.96267169713974}, {'prompt': 'Read the short policy and the facts below. When a fact says that something may be true or false, either case is possible and you are not told which. Answer with a single word on the first line: YES, NO or UNKNOWN.\n\nPolicy: Approval is given if and only if both the dam is certified and the depot is photographed. No other rule applies.\nFacts:\nIt may be true or false that the dam is certified.\nIt is true that the depot is photographed.\nIt is true that the gazebo is reserved.\n\nQuestion: Is approval given?', 'kind': 'C', 'set': 'V2', 'position': 'prima', 'expected_p_UNKNOWN': 0.9046565890312196}, {'prompt': 'Read the short policy and the facts below. When a fact says that something may be true or false, either case is possible and you are not told which. Answer with a single word on the first line: YES, NO or UNKNOWN.\n\nPolicy: Approval is given if and only if both the dam is certified and the depot is photographed. No other rule applies.\nFacts:\nIt is true that the depot is photographed.\nIt is true that the gazebo is reserved.\nIt may be true or false that the dam is certified.\n\nQuestion: Is approval given?', 'kind': 'C', 'set': 'V2', 'position': 'ultima', 'expected_p_UNKNOWN': 0.9998401999473572}, {'prompt': 'Use only the supplied fictional policy and facts. Each listed value is allowed; one actual value per field is chosen independently from its allowed values. Return exactly one of YES, NO, UNKNOWN on the first line, with no explanation.\n\n[R1] In this fictional policy, approval is granted if and only if banner_swept is at least 300. No other approval rule applies.\n[R2] banner_swept = 302.\nlantern_registered = true.\n\nQuestion: Is approval granted?', 'kind': 'A', 'set': 'V3', 'position': 'prima', 'expected_p_UNKNOWN': 3.266253370201589e-13}, {'prompt': 'Use only the supplied fictional policy and facts. Each listed value is allowed; one actual value per field is chosen independently from its allowed values. Return exactly one of YES, NO, UNKNOWN on the first line, with no explanation.\n\n[R1] In this fictional policy, approval is granted if and only if banner_swept is at least 300. No other approval rule applies.\n[R2] lantern_registered = true.\nbanner_swept = 302.\n\nQuestion: Is approval granted?', 'kind': 'A', 'set': 'V3', 'position': 'ultima', 'expected_p_UNKNOWN': 9.357946817618869e-14}, {'prompt': 'Use only the supplied fictional policy and facts. Each listed value is allowed; one actual value per field is chosen independently from its allowed values. Return exactly one of YES, NO, UNKNOWN on the first line, with no explanation.\n\n[R1] In this fictional policy, approval is granted if and only if banner_swept is at least 300. No other approval rule applies.\n[R2] banner_swept = 301 or 302.\nlantern_registered = true.\n\nQuestion: Is approval granted?', 'kind': 'B', 'set': 'V3', 'position': 'prima', 'expected_p_UNKNOWN': 1.522846027057767e-08}, {'prompt': 'Use only the supplied fictional policy and facts. Each listed value is allowed; one actual value per field is chosen independently from its allowed values. Return exactly one of YES, NO, UNKNOWN on the first line, with no explanation.\n\n[R1] In this fictional policy, approval is granted if and only if banner_swept is at least 300. No other approval rule applies.\n[R2] lantern_registered = true.\nbanner_swept = 301 or 302.\n\nQuestion: Is approval granted?', 'kind': 'B', 'set': 'V3', 'position': 'ultima', 'expected_p_UNKNOWN': 2.3820999217605282e-07}, {'prompt': 'Use only the supplied fictional policy and facts. Each listed value is allowed; one actual value per field is chosen independently from its allowed values. Return exactly one of YES, NO, UNKNOWN on the first line, with no explanation.\n\n[R1] In this fictional policy, approval is granted if and only if banner_swept is at least 300. No other approval rule applies.\n[R2] banner_swept = 299 or 300.\nlantern_registered = true.\n\nQuestion: Is approval granted?', 'kind': 'C', 'set': 'V3', 'position': 'prima', 'expected_p_UNKNOWN': 0.0147597612813115}, {'prompt': 'Use only the supplied fictional policy and facts. Each listed value is allowed; one actual value per field is chosen independently from its allowed values. Return exactly one of YES, NO, UNKNOWN on the first line, with no explanation.\n\n[R1] In this fictional policy, approval is granted if and only if banner_swept is at least 300. No other approval rule applies.\n[R2] lantern_registered = true.\nbanner_swept = 299 or 300.\n\nQuestion: Is approval granted?', 'kind': 'C', 'set': 'V3', 'position': 'ultima', 'expected_p_UNKNOWN': 0.98831444978714}], 'mistral': [{'prompt': 'Use only the supplied fictional policy and facts. Each listed value is allowed; one actual value per field is chosen independently from its allowed values. Return exactly one of YES, NO, UNKNOWN on the first line, with no explanation.\n\n[R1] In this fictional policy, approval is granted if and only if (canoe_photographed AND violin_registered). No other approval rule applies.\n[R2] canoe_photographed = true.\nviolin_registered = false.\nladder_insured = true.\n\nQuestion: Is approval granted?', 'kind': 'A', 'set': 'ADD', 'position': 'prima', 'expected_p_UNKNOWN': 0.0004182185803074}, {'prompt': 'Use only the supplied fictional policy and facts. Each listed value is allowed; one actual value per field is chosen independently from its allowed values. Return exactly one of YES, NO, UNKNOWN on the first line, with no explanation.\n\n[R1] In this fictional policy, approval is granted if and only if (canoe_photographed AND violin_registered). No other approval rule applies.\n[R2] violin_registered = false.\nladder_insured = true.\ncanoe_photographed = true.\n\nQuestion: Is approval granted?', 'kind': 'A', 'set': 'ADD', 'position': 'ultima', 'expected_p_UNKNOWN': 0.0027443498838692}, {'prompt': 'Use only the supplied fictional policy and facts. Each listed value is allowed; one actual value per field is chosen independently from its allowed values. Return exactly one of YES, NO, UNKNOWN on the first line, with no explanation.\n\n[R1] In this fictional policy, approval is granted if and only if (canoe_photographed AND violin_registered). No other approval rule applies.\n[R2] canoe_photographed = false or true.\nviolin_registered = false.\nladder_insured = true.\n\nQuestion: Is approval granted?', 'kind': 'B', 'set': 'ADD', 'position': 'prima', 'expected_p_UNKNOWN': 0.0054638744331896}, {'prompt': 'Use only the supplied fictional policy and facts. Each listed value is allowed; one actual value per field is chosen independently from its allowed values. Return exactly one of YES, NO, UNKNOWN on the first line, with no explanation.\n\n[R1] In this fictional policy, approval is granted if and only if (canoe_photographed AND violin_registered). No other approval rule applies.\n[R2] violin_registered = false.\nladder_insured = true.\ncanoe_photographed = false or true.\n\nQuestion: Is approval granted?', 'kind': 'B', 'set': 'ADD', 'position': 'ultima', 'expected_p_UNKNOWN': 0.0038728141225874}, {'prompt': 'Use only the supplied fictional policy and facts. Each listed value is allowed; one actual value per field is chosen independently from its allowed values. Return exactly one of YES, NO, UNKNOWN on the first line, with no explanation.\n\n[R1] In this fictional policy, approval is granted if and only if (canoe_photographed AND violin_registered). No other approval rule applies.\n[R2] canoe_photographed = false or true.\nviolin_registered = true.\nladder_insured = true.\n\nQuestion: Is approval granted?', 'kind': 'C', 'set': 'ADD', 'position': 'prima', 'expected_p_UNKNOWN': 0.0109398877248168}, {'prompt': 'Use only the supplied fictional policy and facts. Each listed value is allowed; one actual value per field is chosen independently from its allowed values. Return exactly one of YES, NO, UNKNOWN on the first line, with no explanation.\n\n[R1] In this fictional policy, approval is granted if and only if (canoe_photographed AND violin_registered). No other approval rule applies.\n[R2] violin_registered = true.\nladder_insured = true.\ncanoe_photographed = false or true.\n\nQuestion: Is approval granted?', 'kind': 'C', 'set': 'ADD', 'position': 'ultima', 'expected_p_UNKNOWN': 0.0174086336046457}, {'prompt': 'Use only the supplied fictional policy and facts. Each listed value is allowed; one actual value per field is chosen independently from its allowed values. Return exactly one of YES, NO, UNKNOWN on the first line, with no explanation.\n\n[R1] In this fictional policy, approval is granted if and only if NOT (easel_insured AND greenhouse_cleared). No other approval rule applies.\n[R2] easel_insured = false.\ngreenhouse_cleared = false.\nescalator_repainted = false.\n\nQuestion: Is approval granted?', 'kind': 'A', 'set': 'V1', 'position': 'prima', 'expected_p_UNKNOWN': 0.0017015397315844}, {'prompt': 'Use only the supplied fictional policy and facts. Each listed value is allowed; one actual value per field is chosen independently from its allowed values. Return exactly one of YES, NO, UNKNOWN on the first line, with no explanation.\n\n[R1] In this fictional policy, approval is granted if and only if NOT (easel_insured AND greenhouse_cleared). No other approval rule applies.\n[R2] greenhouse_cleared = false.\nescalator_repainted = false.\neasel_insured = false.\n\nQuestion: Is approval granted?', 'kind': 'A', 'set': 'V1', 'position': 'ultima', 'expected_p_UNKNOWN': 0.0043211649172008}, {'prompt': 'Use only the supplied fictional policy and facts. Each listed value is allowed; one actual value per field is chosen independently from its allowed values. Return exactly one of YES, NO, UNKNOWN on the first line, with no explanation.\n\n[R1] In this fictional policy, approval is granted if and only if NOT (easel_insured AND greenhouse_cleared). No other approval rule applies.\n[R2] easel_insured = false or true.\ngreenhouse_cleared = false.\nescalator_repainted = false.\n\nQuestion: Is approval granted?', 'kind': 'B', 'set': 'V1', 'position': 'prima', 'expected_p_UNKNOWN': 0.0118034137412905}, {'prompt': 'Use only the supplied fictional policy and facts. Each listed value is allowed; one actual value per field is chosen independently from its allowed values. Return exactly one of YES, NO, UNKNOWN on the first line, with no explanation.\n\n[R1] In this fictional policy, approval is granted if and only if NOT (easel_insured AND greenhouse_cleared). No other approval rule applies.\n[R2] greenhouse_cleared = false.\nescalator_repainted = false.\neasel_insured = false or true.\n\nQuestion: Is approval granted?', 'kind': 'B', 'set': 'V1', 'position': 'ultima', 'expected_p_UNKNOWN': 0.0262720473110675}, {'prompt': 'Use only the supplied fictional policy and facts. Each listed value is allowed; one actual value per field is chosen independently from its allowed values. Return exactly one of YES, NO, UNKNOWN on the first line, with no explanation.\n\n[R1] In this fictional policy, approval is granted if and only if NOT (easel_insured AND greenhouse_cleared). No other approval rule applies.\n[R2] easel_insured = false or true.\ngreenhouse_cleared = true.\nescalator_repainted = false.\n\nQuestion: Is approval granted?', 'kind': 'C', 'set': 'V1', 'position': 'prima', 'expected_p_UNKNOWN': 0.0077380277216434}, {'prompt': 'Use only the supplied fictional policy and facts. Each listed value is allowed; one actual value per field is chosen independently from its allowed values. Return exactly one of YES, NO, UNKNOWN on the first line, with no explanation.\n\n[R1] In this fictional policy, approval is granted if and only if NOT (easel_insured AND greenhouse_cleared). No other approval rule applies.\n[R2] greenhouse_cleared = true.\nescalator_repainted = false.\neasel_insured = false or true.\n\nQuestion: Is approval granted?', 'kind': 'C', 'set': 'V1', 'position': 'ultima', 'expected_p_UNKNOWN': 0.0286359656602144}, {'prompt': 'Read the short policy and the facts below. When a fact says that something may be true or false, either case is possible and you are not told which. Answer with a single word on the first line: YES, NO or UNKNOWN.\n\nPolicy: Approval is given if and only if both the dam is certified and the depot is photographed. No other rule applies.\nFacts:\nIt is true that the dam is certified.\nIt is false that the depot is photographed.\nIt is true that the gazebo is reserved.\n\nQuestion: Is approval given?', 'kind': 'A', 'set': 'V2', 'position': 'prima', 'expected_p_UNKNOWN': 0.3467460572719574}, {'prompt': 'Read the short policy and the facts below. When a fact says that something may be true or false, either case is possible and you are not told which. Answer with a single word on the first line: YES, NO or UNKNOWN.\n\nPolicy: Approval is given if and only if both the dam is certified and the depot is photographed. No other rule applies.\nFacts:\nIt is false that the depot is photographed.\nIt is true that the gazebo is reserved.\nIt is true that the dam is certified.\n\nQuestion: Is approval given?', 'kind': 'A', 'set': 'V2', 'position': 'ultima', 'expected_p_UNKNOWN': 0.666259229183197}, {'prompt': 'Read the short policy and the facts below. When a fact says that something may be true or false, either case is possible and you are not told which. Answer with a single word on the first line: YES, NO or UNKNOWN.\n\nPolicy: Approval is given if and only if both the dam is certified and the depot is photographed. No other rule applies.\nFacts:\nIt may be true or false that the dam is certified.\nIt is false that the depot is photographed.\nIt is true that the gazebo is reserved.\n\nQuestion: Is approval given?', 'kind': 'B', 'set': 'V2', 'position': 'prima', 'expected_p_UNKNOWN': 0.0437848083674907}, {'prompt': 'Read the short policy and the facts below. When a fact says that something may be true or false, either case is possible and you are not told which. Answer with a single word on the first line: YES, NO or UNKNOWN.\n\nPolicy: Approval is given if and only if both the dam is certified and the depot is photographed. No other rule applies.\nFacts:\nIt is false that the depot is photographed.\nIt is true that the gazebo is reserved.\nIt may be true or false that the dam is certified.\n\nQuestion: Is approval given?', 'kind': 'B', 'set': 'V2', 'position': 'ultima', 'expected_p_UNKNOWN': 0.9634582996368408}, {'prompt': 'Read the short policy and the facts below. When a fact says that something may be true or false, either case is possible and you are not told which. Answer with a single word on the first line: YES, NO or UNKNOWN.\n\nPolicy: Approval is given if and only if both the dam is certified and the depot is photographed. No other rule applies.\nFacts:\nIt may be true or false that the dam is certified.\nIt is true that the depot is photographed.\nIt is true that the gazebo is reserved.\n\nQuestion: Is approval given?', 'kind': 'C', 'set': 'V2', 'position': 'prima', 'expected_p_UNKNOWN': 0.9831169247627258}, {'prompt': 'Read the short policy and the facts below. When a fact says that something may be true or false, either case is possible and you are not told which. Answer with a single word on the first line: YES, NO or UNKNOWN.\n\nPolicy: Approval is given if and only if both the dam is certified and the depot is photographed. No other rule applies.\nFacts:\nIt is true that the depot is photographed.\nIt is true that the gazebo is reserved.\nIt may be true or false that the dam is certified.\n\nQuestion: Is approval given?', 'kind': 'C', 'set': 'V2', 'position': 'ultima', 'expected_p_UNKNOWN': 0.9933961629867554}, {'prompt': 'Use only the supplied fictional policy and facts. Each listed value is allowed; one actual value per field is chosen independently from its allowed values. Return exactly one of YES, NO, UNKNOWN on the first line, with no explanation.\n\n[R1] In this fictional policy, approval is granted if and only if banner_swept is at least 300. No other approval rule applies.\n[R2] banner_swept = 302.\nlantern_registered = true.\n\nQuestion: Is approval granted?', 'kind': 'A', 'set': 'V3', 'position': 'prima', 'expected_p_UNKNOWN': 1.1608512068050914e-05}, {'prompt': 'Use only the supplied fictional policy and facts. Each listed value is allowed; one actual value per field is chosen independently from its allowed values. Return exactly one of YES, NO, UNKNOWN on the first line, with no explanation.\n\n[R1] In this fictional policy, approval is granted if and only if banner_swept is at least 300. No other approval rule applies.\n[R2] lantern_registered = true.\nbanner_swept = 302.\n\nQuestion: Is approval granted?', 'kind': 'A', 'set': 'V3', 'position': 'ultima', 'expected_p_UNKNOWN': 3.126783212792361e-06}, {'prompt': 'Use only the supplied fictional policy and facts. Each listed value is allowed; one actual value per field is chosen independently from its allowed values. Return exactly one of YES, NO, UNKNOWN on the first line, with no explanation.\n\n[R1] In this fictional policy, approval is granted if and only if banner_swept is at least 300. No other approval rule applies.\n[R2] banner_swept = 301 or 302.\nlantern_registered = true.\n\nQuestion: Is approval granted?', 'kind': 'B', 'set': 'V3', 'position': 'prima', 'expected_p_UNKNOWN': 0.0047220704145729}, {'prompt': 'Use only the supplied fictional policy and facts. Each listed value is allowed; one actual value per field is chosen independently from its allowed values. Return exactly one of YES, NO, UNKNOWN on the first line, with no explanation.\n\n[R1] In this fictional policy, approval is granted if and only if banner_swept is at least 300. No other approval rule applies.\n[R2] lantern_registered = true.\nbanner_swept = 301 or 302.\n\nQuestion: Is approval granted?', 'kind': 'B', 'set': 'V3', 'position': 'ultima', 'expected_p_UNKNOWN': 0.0003805610467679}, {'prompt': 'Use only the supplied fictional policy and facts. Each listed value is allowed; one actual value per field is chosen independently from its allowed values. Return exactly one of YES, NO, UNKNOWN on the first line, with no explanation.\n\n[R1] In this fictional policy, approval is granted if and only if banner_swept is at least 300. No other approval rule applies.\n[R2] banner_swept = 299 or 300.\nlantern_registered = true.\n\nQuestion: Is approval granted?', 'kind': 'C', 'set': 'V3', 'position': 'prima', 'expected_p_UNKNOWN': 0.011465910822153}, {'prompt': 'Use only the supplied fictional policy and facts. Each listed value is allowed; one actual value per field is chosen independently from its allowed values. Return exactly one of YES, NO, UNKNOWN on the first line, with no explanation.\n\n[R1] In this fictional policy, approval is granted if and only if banner_swept is at least 300. No other approval rule applies.\n[R2] lantern_registered = true.\nbanner_swept = 299 or 300.\n\nQuestion: Is approval granted?', 'kind': 'C', 'set': 'V3', 'position': 'ultima', 'expected_p_UNKNOWN': 0.0103944884613156}]}}
SQUAD_BLOB = """
...
"""

In [ ]:
# 2. Preflight dei dati, senza GPU.
assert MODALITA in ("prova", "completo"), MODALITA
SMOKE = MODALITA == "prova"
DATASET = build_all()
SQUAD = unpack(SQUAD_BLOB)
NEW_DATA_SHA = hashlib.sha256("\n".join(r["prompt"] for r in DATASET).encode()).hexdigest()
SQUAD_SHA = hashlib.sha256("\n".join(r["sq_id"] + r["prompt"] for r in SQUAD).encode()).hexdigest()
assert SQUAD_SHA == "5ac6e6e0ebd4ceedc8c230d8510f025720a41acf3a29c5dbb61e1f7f7fdf9d0b"
assert NEW_DATA_SHA != "22838a1c1c2ed604af004ab00ee48d3603c96afed21847268dadd5434645d33d"
assert len({r["prompt"] for r in DATASET}) == len(DATASET) == 10080
BENCHMARK, SCREEN, HOLDOUT, LEXICAL = make_inputs(DATASET, SMOKE)
if not SMOKE:
    assert (len(BENCHMARK), len(SCREEN), len(HOLDOUT), len(LEXICAL)) == (6960, 720, 600, 1440)
print("Modalità", MODALITA, "| dataset", NEW_DATA_SHA[:16], "| SQuAD", SQUAD_SHA[:16])
print(pd.Series([r["set"] for r in BENCHMARK]).value_counts().to_string())
print(f"Screening {len(SCREEN)} | controllo or {len(LEXICAL)} | POL fino a {len(HOLDOUT) * 2}")

## Inferenza ed esportazione

p(UNKNOWN) è la somma delle probabilità dei primi token delle varianti usate nel Test 4, sul vocabolario intero. La risposta effettivamente generata si misura separatamente. Una quota di massa totale sotto 0,90 superiore al 5% viene segnalata.

Nella raccolta completa il notebook ferma il calcolo degli esiti se falliscono i controlli sulle proiezioni SQuAD o sull’AUROC dei tre modelli con riferimenti Test 4. Qwen e Mistral verificano anche 24 prompt del Test 6. Le coppie B/C con lunghezze diverse in ADD, V1, V2 e POL vengono registrate tra gli scarti. Per HO1 e HO2 si impone la parità di lunghezza nei rispettivi confronti.

Il file testU_risultati.zip contiene rapporto, esiti JSON, benchmark e punteggi per prompt. Gli ZIP per modello permettono di recuperare i risultati anche se la sessione si interrompe prima dell’ultimo modello.

In [ ]:
if AVVIA_GPU:
    import gc
    import inspect
    import time
    import zipfile

    import torch
    import transformers
    from huggingface_hub import HfApi, get_token
    from huggingface_hub.errors import GatedRepoError, HfHubHTTPError
    from scipy.stats import spearmanr
    from transformers import AutoModelForCausalLM, AutoTokenizer

    def hf_token():
        token = get_token()
        if token is None:
            try:
                from google.colab import userdata
                token = userdata.get("HF_TOKEN")
            except Exception:
                token = None
        return token

    def batch_inputs(tokenizer, model_key, prompts, device):
        options = {"enable_thinking": False} if model_key == "qwen" else {}
        texts = [
            tokenizer.apply_chat_template([{"role": "user", "content": p}], tokenize=False,
                                          add_generation_prompt=True, **options)
            for p in prompts
        ]
        return tokenizer(texts, padding=True, return_tensors="pt", add_special_tokens=False).to(device)

    def label_ids(tokenizer):
        audit = {}
        for label in LABELS:
            variants = [prefix + word for word in (label, label.capitalize(), label.lower()) for prefix in ("", " ")]
            audit[label] = {v: tokenizer.encode(v, add_special_tokens=False) for v in variants}
        ids = {label: sorted({sequence[0] for sequence in audit[label].values()}) for label in LABELS}
        if len(set().union(*map(set, ids.values()))) != sum(map(len, ids.values())):
            raise RuntimeError(f"Il tokenizer condivide i primi token fra etichette: {ids}")
        return ids, audit

    def batches(rows, size):
        for start in range(0, len(rows), size):
            yield rows[start:start + size]

    def predict(model, tokenizer, key, rows, ids, size):
        for batch in batches(rows, size):
            enc = batch_inputs(tokenizer, key, [r["prompt"] for r in batch], model.device)
            with torch.inference_mode():
                result = model.generate(**enc, do_sample=False, max_new_tokens=8, use_cache=True,
                                        return_dict_in_generate=True, output_logits=True,
                                        pad_token_id=tokenizer.pad_token_id)
                logits = result.logits[0].float()
                if not bool(torch.isfinite(logits).all()):
                    raise RuntimeError("Logit non finiti al primo passo")
                p = logits.softmax(dim=-1)
                scores = torch.stack([p[:, ids[label]].sum(dim=-1) for label in LABELS], dim=-1).cpu().tolist()
                completions = tokenizer.batch_decode(
                    result.sequences[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
            lengths = enc["attention_mask"].sum(dim=-1).cpu().tolist()
            for row, text, probability, n in zip(batch, completions, scores, lengths, strict=True):
                yield dict(row, generated=text, parsed=parse_label(text), prompt_tokens=n,
                           mass_total=sum(probability), **{f"p_{k}": v for k, v in zip(LABELS, probability, strict=True)})
            del result, logits, p

    def hidden_last(model, tokenizer, key, rows, layers, size):
        blocks = []
        for batch in batches(rows, size):
            enc = batch_inputs(tokenizer, key, [r["prompt"] for r in batch], model.device)
            pos = (enc["attention_mask"].cumsum(dim=-1) - 1).clamp_min(0)
            with torch.inference_mode():
                out = model.base_model(**enc, position_ids=pos, use_cache=False, output_hidden_states=True)
                vectors = torch.stack([out.hidden_states[l][:, -1] for l in layers], dim=1)
            blocks.append(vectors.float().cpu().numpy())
            del out, vectors
        return np.concatenate(blocks, axis=0)

    def squad_detector(model, tokenizer, key, rows, layers, size, smoke, reference):
        train = [r for r in rows if r["part"] == "train"]
        test = [r for r in rows if r["part"] == "test"]
        if smoke:
            train = [r for y in (0, 1) for r in [x for x in train if x["unanswerable"] == y][:20]]
            test = [r for y in (0, 1) for r in [x for x in test if x["unanswerable"] == y][:20]]
        x = hidden_last(model, tokenizer, key, train, layers, size).astype(np.float64)
        y = np.array([r["unanswerable"] for r in train])
        direction = x[y == 1].mean(axis=0) - x[y == 0].mean(axis=0)
        norms = np.linalg.norm(direction, axis=-1)
        if not np.isfinite(norms).all() or (norms <= 1e-12).any():
            raise RuntimeError("Direzione SQuAD non valida")
        direction /= norms[:, None]
        train_scores = np.einsum("nld,ld->nl", x, direction)
        mu, std = train_scores.mean(axis=0), train_scores.std(axis=0)
        del x, train_scores
        def project(batch):
            h = hidden_last(model, tokenizer, key, batch, layers, size)
            return np.einsum("nld,ld->nl", h, direction)
        scores = project(test)
        if not np.isfinite(scores).all() or (std <= 1e-12).any():
            raise RuntimeError("Proiezione SQuAD non valida")
        ytest = [r["unanswerable"] for r in test]
        auc = [float(roc_auc_score(ytest, scores[:, j])) for j in range(len(layers))]
        result = dict(train=len(train), test=len(test), layers=layers, auroc_per_strato=auc,
                      auroc_aggregata=float(roc_auc_score(ytest, ((scores - mu) / std).mean(axis=1))),
                      riferimento_scala="media e deviazione standard per strato su SQuAD train",
                      allineamento="NON APPLICABILE" if reference is None else "SOLO RUN COMPLETA")
        if reference is not None and not smoke:
            by_id = {r["sq_id"]: i for i, r in enumerate(test)}
            actual = scores[[by_id[q] for q in reference["ids"]]]
            expected = np.array(reference["projections"])
            # Decide la correlazione di Pearson (≥ 0.999 per strato) e ΔAUROC ≤ 0.01. Spearman su 20 punti si riporta soltanto:
            # basta lo scambio di due valori vicini per scendere a 0.9985, anche con differenze numeriche minime.
            pearson = [float(np.corrcoef(actual[:, j], expected[:, j])[0, 1]) for j in range(len(layers))]
            rho = [float(spearmanr(actual[:, j], expected[:, j]).statistic) for j in range(len(layers))]
            delta_auc = (np.array(auc) - reference["auc"]).tolist()
            max_error = float(np.max(np.abs(actual - expected)))
            print(f"  SQuAD allineamento {key}: pearson={pearson}, spearman={rho}, delta AUROC={delta_auc}, "
                  f"max |delta proiezione|={max_error:.4g}")
            if not np.isfinite(pearson).all() or min(pearson) < .999 or max(map(abs, delta_auc)) > .01:
                raise RuntimeError(f"{key}: allineamento SQuAD con Test 4 non riuscito. Esiti non calcolati.")
            result["allineamento"] = dict(pearson=pearson, spearman=rho, delta_auc=delta_auc, errore_assoluto_max=max_error)
        return project, mu, std, result

    def prompt_lengths(tokenizer, key, rows):
        out = []
        for chunk in batches(rows, 128):
            enc = batch_inputs(tokenizer, key, [r["prompt"] for r in chunk], "cpu")
            out.extend(enc["attention_mask"].sum(dim=-1).tolist())
        return out

    def checkpoint(folder, stage, rows, compute, size=128):
        sig = fingerprint(rows)[:16]
        folder = folder / "checkpoint" / f"{stage}-{sig}"
        folder.mkdir(parents=True, exist_ok=True)
        out = []
        for i, chunk in enumerate(batches(rows, size)):
            path = folder / f"{i:04d}.json"
            expected = [r["row_id"] for r in chunk]
            if path.exists():
                part = json.loads(path.read_text())
            else:
                part = list(compute(chunk))
                if [r["row_id"] for r in part] != expected:
                    raise RuntimeError(f"Risultati incompleti in {stage}, blocco {i}")
                write_json(path, part)
            if [r["row_id"] for r in part] != expected:
                raise RuntimeError(f"Checkpoint incoerente in {stage}, blocco {i}")
            out.extend(part)
            if i == 0 or (i + 1) % 5 == 0 or len(out) == len(rows):
                print(f"  {stage} {len(out)}/{len(rows)}", flush=True)
        return out

    def export_model(folder, result, tables):
        folder.mkdir(parents=True, exist_ok=True)
        write_json(folder / "esito.json", result)
        for name, rows in tables.items():
            path = folder / f"{name}.csv"
            tmp = path.with_suffix(".csv.tmp")
            pd.json_normalize(rows).to_csv(tmp, index=False)
            os.replace(tmp, path)
        names = ["esito.json", *(f"{name}.csv" for name in tables)]
        archive = folder / f"{folder.name}.zip"
        tmp = archive.with_suffix(".zip.tmp")
        with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as z:
            for n in names:
                z.write(folder / n, n)
        os.replace(tmp, archive)
        return archive

In [ ]:
def evaluate_model(key, config, root, smoke, token):
    model_id = config["id"]
    revision = config["sha"]
    if revision is None:
        revision_path = root / "llama_revision.json"
        if revision_path.is_file():
            revision = json.loads(revision_path.read_text())["sha"]
        else:
            revision = HfApi().model_info(model_id, token=token).sha
            write_json(revision_path, {"sha": revision})
    manifest = dict(version="U-v1-notebook-2", modello=model_id, sha=revision, dtype="bfloat16", attention="eager",
                    batch_size=BATCH_SIZE, squad_batch_size=SQUAD_BATCH_SIZE, transformers=transformers.__version__,
                    torch=str(torch.__version__), gpu=torch.cuda.get_device_name(0), modalita=MODALITA,
                    seme=20261120, dataset=NEW_DATA_SHA, squad=SQUAD_SHA)
    folder = root / f"{key}-{fingerprint(manifest)[:12]}"
    folder.mkdir(parents=True, exist_ok=True)
    if (folder / f"{folder.name}.zip").is_file():
        print(f"{key}: risultati già completati")
        return folder

    tokenizer = AutoTokenizer.from_pretrained(model_id, revision=revision, token=token, trust_remote_code=False,
                                              **({"fix_mistral_regex": True} if key == "mistral" else {}))
    tokenizer.padding_side = "left"
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    ids, audit = label_ids(tokenizer)
    if config.get("label_ids") is not None and ids != config["label_ids"]:
        raise RuntimeError(f"{key}: token delle etichette diversi dal Test 4")
    model = AutoModelForCausalLM.from_pretrained(model_id, revision=revision, token=token, dtype=torch.bfloat16,
                                                 device_map={"": 0}, attn_implementation="eager",
                                                 trust_remote_code=False, low_cpu_mem_usage=True).eval()
    try:
        return evaluate_loaded(key, model, tokenizer, config, folder, smoke, manifest, ids, audit)
    finally:
        del model
        gc.collect()
        torch.cuda.empty_cache()

def evaluate_loaded(key, model, tokenizer, config, folder, smoke, manifest, ids, audit):
    def predictions(rows):
        return predict(model, tokenizer, key, rows, ids, BATCH_SIZE)

    calibration = REFERENCE["outputs"].get(key)
    outputs_alignment = None
    if calibration:
        out = list(predict(model, tokenizer, key, calibration, ids, 1))
        max_delta = max(abs(r["p_UNKNOWN"] - r["expected_p_UNKNOWN"]) for r in out)
        outputs_alignment = dict(n=len(out), errore_assoluto_max=max_delta)
        print(f"  Allineamento Test 6, p(UNKNOWN): max |delta| = {max_delta:.4g}")
        if max_delta > .001:
            raise RuntimeError(f"{key}: allineamento delle uscite con Test 6 non riuscito. Esiti non calcolati.")

    layers = config.get("layers") or [
        round(f * model.config.num_hidden_layers) for f in (.35, .44, .53, .62, .71, .80)
    ]
    project, mu, std, squad = squad_detector(model, tokenizer, key, SQUAD, layers, SQUAD_BATCH_SIZE, smoke,
                                           REFERENCE["squad"].get(key))
    kept, dropped = filter_length(BENCHMARK, prompt_lengths(tokenizer, key, BENCHMARK))
    main = checkpoint(folder, "benchmark", kept, predictions)
    benchmark, by_cell = benchmark_metrics(main, smoke)
    threshold = transferred_threshold(main, smoke)
    threshold_alt = {score: transferred_threshold(main, smoke, score)
                     for score in ("meno_p_massima", "entropia")}

    screen_raw = checkpoint(folder, "screening", SCREEN, predictions)
    screen = accuracy_by_label(screen_raw)
    chosen = pick_candidate(screen)
    polarity = dict(candidato=chosen, esito="NON VALUTABILE", coerenza_AB=None)
    pol_raw = []
    pol_drop = []
    pol_stats = []
    if chosen:
        pol = identify([question_variant(r, p) for r in HOLDOUT for p in ("granted", chosen)])
        pol_keep, pol_drop = filter_length(pol, prompt_lengths(tokenizer, key, pol))
        pol_raw = checkpoint(folder, "polarita", pol_keep, predictions)
        pol_stats, _ = benchmark_metrics(pol_raw, smoke)
        pol_table = pd.DataFrame(pol_raw)
        a_b = pol_table[pol_table.kind.isin(["A", "B"])].copy()
        a_b["key"] = a_b.pair_key.str.rsplit("|", n=1).str[0] + "|" + a_b.kind
        paired = a_b.pivot(index="key", columns="polarity", values="parsed").dropna()
        polarity["coerenza_AB"] = float(paired[chosen].eq(paired.granted.map({"YES": "NO", "NO": "YES"})).mean()) if len(paired) else None
        polarity["n_coppie_coerenza"] = len(paired)
        polarity["accuratezza_AB_test"] = accuracy_by_label(a_b.to_dict("records"))
        polarity["esito"] = next(r["esito"] for r in pol_stats if r["polarity"] == chosen)

    lex_keep, lex_drop = filter_length(LEXICAL, prompt_lengths(tokenizer, key, LEXICAL), lexical=True)
    def score_lexical(rows):
        output = list(predictions(rows))
        proj = project(rows)
        agg = ((proj - mu) / std).mean(axis=1)
        return [dict(r, score_squad=float(s), **{f"score_L{l}": float(v) for l, v in zip(layers, scores, strict=True)})
                for r, s, scores in zip(output, agg, proj, strict=True)]

    lex_raw = checkpoint(folder, "controllo_or", lex_keep, score_lexical)
    lex, lex_gen = lexical_metrics(lex_raw, smoke)
    prompt_rows = main + screen_raw + pol_raw + lex_raw
    low_mass = float(np.mean([r["mass_total"] < .90 for r in prompt_rows]))
    result = dict(**manifest, label_ids=ids, audit_tokenizzazione=audit, allineamento_uscite=outputs_alignment,
                  rilevatore_squad=squad, benchmark=benchmark, soglia=threshold, soglie_descrittive=threshold_alt,
                  controllo_or=lex, or_generazione=lex_gen,
                  screening=screen, polarita=polarity, polarita_benchmark=pol_stats,
                  scarti_prompt={"benchmark": len(dropped), "O": len(lex_drop), "POL": len(pol_drop)},
                  quota_massa_bassa=low_mass, cautela_massa=low_mass > .05)
    archive = export_model(folder, result, dict(punteggi=prompt_rows, celle=by_cell, scarti=dropped + lex_drop + pol_drop))
    print(f"  Salvato {archive.name} | massa < 0.90: {low_mass:.1%}")
    return folder

def export_total(root, folders, skipped, smoke):
    summaries = {key: json.loads((p / "esito.json").read_text()) for key, p in folders.items()}
    tables, rows = [], []
    for key, result in summaries.items():
        for r in result["benchmark"]:
            by_kind = {k["kind"]: k for k in r["comportamento"]}
            rows.append(dict(modello=key, insieme=r["set"], W=r["W"]["stima"], IC95=r["W"]["ci95"],
                             AUROC=r["AUROC"], coppie=r["W"]["coppie"], HU1=r["esito"],
                             W_senza_UNKNOWN=r["W_senza_UNKNOWN_generato"]["stima"],
                             quota_UNKNOWN_C=by_kind.get("C", {}).get("quota_unknown"),
                             quota_UNKNOWN_B=by_kind.get("B", {}).get("quota_unknown"),
                             acc_A=by_kind.get("A", {}).get("acc"), acc_B=by_kind.get("B", {}).get("acc")))
        table = pd.read_csv(folders[key] / "punteggi.csv", keep_default_na=False)
        table.insert(0, "modello", key)
        tables.append(table)
    scorecard = []
    for hypothesis, formats in (("HU1", ("ADD", "V1", "V2")), ("HU2", ("V1", "V2"))):
        for fmt in formats:
            section = "benchmark" if hypothesis == "HU1" else "soglia"
            confirmed = [k for k, r in summaries.items()
                         if any(x["set"] == fmt and x["esito"] == "CONFERMATA" for x in r[section])]
            scorecard.append(dict(ipotesi=hypothesis, insieme=fmt, confermati=confirmed, denominatore=4,
                                  criterio_3_su_4=None if smoke else len(confirmed) >= 3,
                                  n_modelli_completati=len(summaries)))
    report = [
        f"Test U | modalità {MODALITA} | modelli completati {len(summaries)}/4",
        "La modalità prova verifica la pipeline e non produce esiti scientifici." if smoke
        else "La regola per la libreria richiede almeno 3 conferme fra i 4 modelli previsti, per lo stesso formato.",
        f"Modelli saltati: {skipped}",
        pd.DataFrame(rows).round(4).to_string(index=False),
        "",
        pd.DataFrame(scorecard).to_string(index=False),
    ]
    for key, r in summaries.items():
        ho = [x for x in r["controllo_or"] if x["scope"] == "tutte" and x["misura"] == "score_squad" and x["ipotesi"] in ("HO1", "HO2")]
        report.extend([
            "", key, f"  Scarti: {r['scarti_prompt']} | quota massa bassa {r['quota_massa_bassa']:.2%}",
            f"  Soglia: {r['soglia']}",
            f"  HO1 e HO2: {ho}",
            f"  Polarità: {r['polarita']}",
        ])
    write_json(root / "esiti.json", dict(modelli=summaries, sintesi=scorecard, saltati=skipped, modalita=MODALITA))
    pd.DataFrame(rows).to_csv(root / "benchmark.csv", index=False)
    if tables:
        pd.concat(tables, ignore_index=True).to_csv(root / "punteggi.csv", index=False)
    (root / "rapporto.txt").write_text("\n".join(report), encoding="utf-8")
    archive = root / "testU_risultati.zip"
    temp = archive.with_suffix(".zip.tmp")
    with zipfile.ZipFile(temp, "w", zipfile.ZIP_DEFLATED) as z:
        for name in ("esiti.json", "benchmark.csv", "rapporto.txt", "punteggi.csv"):
            if name != "punteggi.csv" or tables:
                z.write(root / name, name)
        for key, folder in folders.items():
            for name in ("celle.csv", "scarti.csv"):
                z.write(folder / name, f"{key}/{name}")
    os.replace(temp, archive)
    print(pd.DataFrame(rows).round(4).to_string(index=False))
    print("ZIP finale:", archive)
    return archive

def run_test():
    if MODALITA not in ("prova", "completo"):
        raise ValueError("MODALITA deve essere 'prova' o 'completo'")
    if not MODELLI or len(set(MODELLI)) != len(MODELLI) or not set(MODELLI).issubset(MODEL_CONFIG):
        raise ValueError("MODELLI deve contenere chiavi diverse fra qwen, mistral, olmo, llama")
    if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():
        raise RuntimeError("Seleziona una GPU A100 in Colab per il protocollo bf16.")
    if MODALITA == "completo" and "A100" not in torch.cuda.get_device_name(0):
        raise RuntimeError("La run completa richiede una GPU A100 per rispettare il protocollo.")
    if USA_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
    base = Path("/content/drive/MyDrive/testU_parziali" if USA_DRIVE else "/content/testU_parziali")
    root = base / MODALITA
    root.mkdir(parents=True, exist_ok=True)
    token = hf_token()
    folders, skipped = {}, {}
    for key in MODELLI:
        print(f"\n{key} | {MODALITA}", flush=True)
        if key == "llama" and not token:
            skipped[key] = "HF_TOKEN assente"
            print(f"  Saltato: {skipped[key]}")
            continue
        try:
            if key == "llama":
                HfApi().auth_check(repo_id=MODEL_CONFIG[key]["id"], token=token)
            folders[key] = evaluate_model(key, MODEL_CONFIG[key], root, MODALITA == "prova", token)
        except HfHubHTTPError as exc:
            code = exc.response.status_code if exc.response is not None else None
            if key != "llama" or (not isinstance(exc, GatedRepoError) and code not in (401, 403)):
                raise
            skipped[key] = "Accesso al checkpoint Llama non autorizzato"
            print(f"  Saltato: {skipped[key]}")
        export_total(root, folders, skipped, MODALITA == "prova")
    return export_total(root, folders, skipped, MODALITA == "prova")

In [ ]:
# 3. Avvio esplicito della prova o della raccolta.
if AVVIA_GPU:
    ARCHIVIO_TEST_U = run_test()
else:
    print("Preflight concluso. Imposta AVVIA_GPU = True per eseguire la modalità selezionata.")

## Cella facoltativa: controllo dei coseni del test 7

Solo CPU, 2–5 minuti, indipendente dalle celle precedenti. Legge `test6_risultati_Qwen3-8B.zip` e `test6_stati_Qwen3-8B.zip` da Drive e salva `controllo_coseni_test7.json` in `MyDrive/testU_parziali/`. Non decide nulla del Test U. Parte solo con `ESEGUI_COSENI = True`, quindi «Esegui tutte» la salta.

In [ ]:
# 11. FACOLTATIVA, solo CPU (2–5 minuti). Controllo dei coseni allo strato 36 di Qwen nel test 7. Non decide nulla.
# Ipotesi da verificare (riepilogo Step 1): i coseni vicini a ±1 verrebbero da poche dimensioni con deviazione standard quasi
# nulla, che pesano molto quando la direzione della sonda si riporta nello spazio originale (divisione per scale_).
# Servono test6_risultati_Qwen3-8B.zip e test6_stati_Qwen3-8B.zip su Drive (gli stessi usati nel test 7, una sola copia).
# Si può eseguire anche da sola: non usa nulla delle celle precedenti. Per eseguirla metti ESEGUI_COSENI = True
# (con False non fa nulla, così «Esegui tutte» non la avvia per sbaglio).
ESEGUI_COSENI = False

def controllo_coseni():
    global ESITO7, T7, RIS7
    import glob, json, os, zipfile
    import numpy as np, pandas as pd
    from pathlib import Path
    from sklearn.decomposition import PCA
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler

    TAG7, STRATI7, K_PCA7, TOP7 = "Qwen3-8B", (32, 36), 64, 10
    ROOT7 = os.environ.get("C7_ROOT", "/content/drive/MyDrive")
    OUT7 = Path(os.environ.get("C7_OUT", "/content/coseni7")); OUT7.mkdir(parents=True, exist_ok=True)
    SAVE7 = Path(os.environ.get("C7_SAVE", "/content/drive/MyDrive/testU_parziali"))
    if ROOT7.startswith("/content/drive") and not os.path.exists("/content/drive/MyDrive"):
        from google.colab import drive; drive.mount("/content/drive")
    # Coseni del test 7 nello spazio originale (rapporto del test 7), per controllare di riprodurre lo stesso calcolo
    ATTESI7 = {36: {"G1~G2": .98, "G1~G3": -.96, "G2~G3": -.98, "T6_ADD~G1": .98, "T6_ADD~G2": .99, "T6_ADD~G3": -.99},
               32: {"G1~G2": .34, "G1~G3": .25, "G2~G3": .50, "T6_ADD~G1": .51, "T6_ADD~G2": .32, "T6_ADD~G3": .02}}

    def trova7(nome):
        hits = sorted({str(Path(p).resolve()) for p in glob.glob(f"{ROOT7}/**/{nome[:-4]}*.zip", recursive=True)})
        assert len(hits) == 1, f"{nome}: trovati {len(hits)} file {hits}. Lasciane una sola copia su Drive."
        return hits[0]
    for nome in (f"test6_risultati_{TAG7}.zip", f"test6_stati_{TAG7}.zip"):
        with zipfile.ZipFile(trova7(nome)) as z:
            z.extractall(OUT7)
    META7 = pd.read_csv(OUT7 / f"metadata_test6_{TAG7}.csv", keep_default_na=False)
    META7["scartata"] = META7["scartata"].astype(str).str.strip().eq("True")
    RI7 = json.loads((OUT7 / f"run_info_test6_{TAG7}.json").read_text())
    S7 = np.load(OUT7 / f"S_fine_{TAG7}.npy", mmap_mode="r")
    GRID7 = RI7["grid"]
    assert S7.shape[0] == len(META7) and S7.shape[1] == len(GRID7) and all(l in GRID7 for l in STRATI7), (S7.shape, GRID7)

    def dev7(sets):
        # come nel test 7: coppie B/C non scartate, ordinate per pair_key e tipo; ogni formulazione pesa uguale
        sub = META7[META7.set.isin(sets) & META7.kind.isin(["B", "C"]) & ~META7.scartata].sort_values(["pair_key", "kind"])
        n = sub.set.value_counts()
        w = np.array([len(sub) / (len(n) * n[s]) for s in sub.set])
        return sub.index.to_numpy(), (sub.kind == "C").to_numpy().astype(int), w

    def direzioni7(sets, l):
        idx, y, w = dev7(sets)
        X = np.asarray(S7[idx, GRID7.index(l)], np.float32)
        sc = StandardScaler().fit(X)
        p = PCA(n_components=K_PCA7, svd_solver="randomized", whiten=True, random_state=0).fit(sc.transform(X))
        b = LogisticRegression(C=1.0, max_iter=3000).fit(p.transform(sc.transform(X)), y, sample_weight=w).coef_[0]
        w_std = p.components_.T @ (b / np.sqrt(p.explained_variance_))     # spazio standardizzato
        w_orig = w_std / sc.scale_                                           # spazio originale (come nel test 7)
        return w_std, w_orig, sc.scale_

    unit = lambda v: v / np.linalg.norm(v)
    GIRI7 = {"T6_ADD": ["ADD"], "G1": ["ADD", "V1"], "G2": ["ADD", "V2"], "G3": ["V1", "V2"]}
    RIS7, righe = {}, []
    for l in STRATI7:
        D = {g: direzioni7(s, l) for g, s in GIRI7.items()}
        scala = np.median(np.stack([D[g][2] for g in D]), axis=0)          # deviazione standard per dimensione (mediana tra i giri)
        piccole = np.argsort(scala)[:TOP7]                                   # le TOP7 dimensioni con deviazione standard minima
        info = {}
        for g, (ws, wo, s) in D.items():
            q = wo ** 2 / (wo ** 2).sum()
            top = np.argsort(q)[::-1][:3]
            info[g] = dict(quota_norma_top1=float(np.sort(q)[-1]), quota_norma_top10=float(np.sort(q)[-10:].sum()),
                           quota_norma_su_dim_a_scala_minima=float(q[piccole].sum()), dimensioni_dominanti=top.tolist(),
                           segno_sulla_dominante=[int(np.sign(wo[i])) for i in top])
        nomi = list(GIRI7)
        for i in range(len(nomi)):
            for j in range(i + 1, len(nomi)):
                a, b = nomi[i], nomi[j]
                wo_a, wo_b = D[a][1].copy(), D[b][1].copy()
                wo_a[piccole] = 0; wo_b[piccole] = 0
                righe.append(dict(strato=l, coppia=f"{a}~{b}",
                                  coseno_originale=float(unit(D[a][1]) @ unit(D[b][1])),
                                  atteso_test7=ATTESI7[l][f"{a}~{b}"],
                                  coseno_standardizzato=float(unit(D[a][0]) @ unit(D[b][0])),
                                  coseno_originale_senza_dim_a_scala_minima=float(unit(wo_a) @ unit(wo_b))))
        RIS7[l] = dict(scala_min=float(scala.min()), scala_mediana=float(np.median(scala)),
                       scala_dim_minime=scala[piccole].tolist(), dim_a_scala_minima=piccole.tolist(), giri=info)
    T7 = pd.DataFrame(righe)
    riprodotto = bool((T7.coseno_originale - T7.atteso_test7).abs().max() <= .02)
    print(f"Riproduzione dei coseni del test 7 (tolleranza 0.02, valori arrotondati nel rapporto): {'OK' if riprodotto else 'NO'}")
    print(T7.round(3).to_string(index=False))
    for l in STRATI7:
        r = RIS7[l]
        print(f"\nStrato {l}: deviazione standard per dimensione, minima {r['scala_min']:.3g}, mediana {r['scala_mediana']:.3g}; "
              f"le {TOP7} minime {np.round(r['scala_dim_minime'], 4).tolist()}")
        for g, v in r["giri"].items():
            print(f"  {g:7s} quota della norma: prima dimensione {v['quota_norma_top1']:.2f}, prime 10 {v['quota_norma_top10']:.2f}, "
                  f"sulle {TOP7} a scala minima {v['quota_norma_su_dim_a_scala_minima']:.2f} | dominanti {v['dimensioni_dominanti']} "
                  f"segni {v['segno_sulla_dominante']}")
    # Lettura fissata prima di eseguire. Se i coseni del test 7 non si riproducono (tolleranza 0.02): NON VALUTABILE. Altrimenti l'ipotesi dell'artefatto è SOSTENUTA se, allo strato 36: (a) le TOP7 dimensioni a scala
    # minima portano più del 90% della norma in ogni giro, e (b) ricalcolati nello spazio standardizzato e senza quelle dimensioni,
    # i coseni non sono più vicini a ±1 (|coseno| < 0.9 in tutte le coppie). NON SOSTENUTA se (a) non vale per nessun giro. Altrimenti NON CHIARA.
    r36 = RIS7[36]["giri"]; t36 = T7[T7.strato == 36]
    a_ogni = all(v["quota_norma_su_dim_a_scala_minima"] > .9 for v in r36.values())
    a_nessuno = not any(v["quota_norma_su_dim_a_scala_minima"] > .9 for v in r36.values())
    b_ok = bool((t36.coseno_standardizzato.abs() < .9).all() and (t36.coseno_originale_senza_dim_a_scala_minima.abs() < .9).all())
    ESITO7 = ("NON VALUTABILE" if not riprodotto else "SOSTENUTA" if a_ogni and b_ok else "NON SOSTENUTA" if a_nessuno else "NON CHIARA")
    print(f"\nIpotesi dell'artefatto allo strato 36: {ESITO7}" + ("" if riprodotto else " (i coseni del test 7 non sono stati riprodotti)"))
    SAVE7.mkdir(parents=True, exist_ok=True)
    (SAVE7 / "controllo_coseni_test7.json").write_text(json.dumps(dict(esito=ESITO7, riprodotto=riprodotto, coseni=righe,
                                                                         dettagli={str(k): v for k, v in RIS7.items()}), indent=1))
    print("Salvato:", SAVE7 / "controllo_coseni_test7.json")


if ESEGUI_COSENI:
    controllo_coseni()
else:
    print("Cella dei coseni saltata (ESEGUI_COSENI = False).")
